[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/throwanalysis_googlecolab/blob/main/generated/OpenCapMono_Pitching_Phase_Morphology_v15_SourceAndOpenCapFix.ipynb)


# OpenCap Monocular × 5-Phase Pitching × Morphology PC1 — v15 — v15 
## Youth baseball pitching — symposium analysis notebook

**Purpose**

This notebook integrates:

1. **Three Google Drive video folders**
   - Elementary school
   - Mixed elementary + junior-high
   - High school
2. **Existing MediaPipe + LightGBM 5-phase classifier**
3. **Video-derived body-proportion morphology**
4. **PCA-derived Morphology PC1 — v15**
5. **OpenCap Monocular / OpenSim kinematics**
6. **Phase-specific normalized waveforms**
7. **Outputs for SPM regression**

### Primary scientific question

> How does the pitching kinematic chain change continuously along a video-derived body-proportion morphology axis?

### Important interpretation

`Morphology_PC1` is **not bone age or skeletal age**.  
It is a continuous **video-derived body-proportion morphology axis**.

Because actual height and body mass are unavailable, OpenCap outputs should be used primarily for **joint kinematics / timing**, not personalized kinetics or muscle-force interpretation.


### v15 — source-folder + OpenCap auxiliary-data fixes

- Fixes the second high-school source to `結合済み_選手ID別` and scans it recursively.
- Test mode now selects **1 video per configured source** (ZIP or folder), rather than 3 per group.
- Adds bounded fallback search for a missing source path by exact folder basename.
- Ensures official WHAM auxiliary data includes `model_transfer/smplx_to_smpl.pkl`; downloads/re-extracts the official auxiliary archive when any required file is missing.
- Patches OpenCap's ffmpeg trim filter to pad odd video dimensions to even numbers for H.264 encoding.
- Makes SPM export safe when upstream OpenCap outputs are empty.
- Adds an **Open in Colab** badge linked directly to this GitHub notebook.


### v14 — MediaPipe installation fix for current Colab / Python 3.13

- Removes the host-side `scikit-learn==1.4.2` pin that forced a source build on Python 3.13 and caused the combined pip command to fail.
- Installs `mediapipe==0.10.35` **separately and wheel-only**, so another package cannot hide its installation status.
- Verifies `import mediapipe` immediately after installation.
- Keeps the existing LightGBM PKL workflow; its sklearn-version warning is suppressed only after validating the expected 23 features/classes.
- Prints host Python and package versions for reproducibility.


### v13 — intrinsics compatibility + fixed PKL priority path

- Fixes the observed OpenCap failure `ModuleNotFoundError: numpy._core` when optimization loads the generated intrinsics pickle.
- Intrinsics pickle is now created **inside the OpenCap micromamba environment (NumPy 1.21.6)**, not by Colab host NumPy 2.x.
- Adds a compatibility probe that immediately loads the generated intrinsics with OpenCap's own `load_intrinsics()` before batch processing.
- Prioritizes `/content/drive/MyDrive/臨床研究_発表/投球動作解析/throwdetect_trained_lgb_model_20250108_5phase.pkl` exactly.
- Pins host scikit-learn to 1.4.2 to match the phase-model pickle and remove the version warning.
- Gives empty OpenSim outputs fixed schemas and prevents downstream `KeyError: 'filename'`.
- Can reuse a completed WHAM result on rerun instead of spending ~6 minutes recomputing it.


### v12 — speed + phase-model auto-detect + OpenCap visualization fix

- Auto-detects `throwdetect_trained_lgb_model_20250108_5phase.pkl`, including the clinical-research Drive folder.
- MediaPipe inference is capped at a target sampling rate (default **30 Hz**) while preserving the original video timeline.
- Keeps Pose Landmarker **Full** by default for LightGBM compatibility; `lite` is a one-line option for validation.
- Batches LightGBM prediction once per video instead of once per frame.
- QC video reuses landmarks from the first MediaPipe pass; **no second MediaPipe inference**.
- QC video generation can be disabled for the 400-video production run.
- Disables WHAM/PyTorch3D visualization that caused the observed `ModuleNotFoundError: pytorch3d` in the Slim installation.
- Disables optional optimization plotting for batch speed/stability while preserving numerical OpenSim outputs.


### v8 diagnostic fixes

- Replaces deprecated 2D `np.cross` with an equivalent scalar determinant.
- Strictly verifies OpenCap imports and **initializes WHAM before batch processing**.
- Adds the 133-keypoint ViTPose-H whole-body checkpoint at the path expected by the current OpenCap fork.
- Uses explicit approximate per-video intrinsics instead of a missing iPhone fallback.
- Captures and saves the real OpenCap stdout/traceback for every trial.
- Stops after the first OpenCap failure during debugging.
- Prevents downstream empty-waveform errors from hiding the upstream failure.


### v10 — OpenCap environment compatibility fixes\n\n- Keeps OpenCap Slim's `mmcv-full==1.7.2`.\n- Patches only the stale ViTPose/mmpose MMCV upper-bound guard (`1.5.0 → 1.7.2`).\n- Runs all micromamba subprocesses from the OpenCap repository root.\n- Adds the repository root to `PYTHONPATH`, matching the Docker deployment model.\n- Separately verifies `optimization`, `WHAM.demo`, required assets, and `initialize_wham()`.\n- Stops with a precise preflight error before processing videos if a real incompatibility remains.\n

### v9 — SMPL ZIP auto-detect fix

- Tries the user-reported Drive-root path first.
- Also checks standard `/content/drive/MyDrive` locations.
- Detects variants such as `SMPL_python_v.1.1.0(1).zip`.
- Uses a bounded fallback search instead of scanning the whole Drive indefinitely.
- Removes the blocking `files.upload()` fallback for SMPL.
- Validates the ZIP contents before continuing.
- Extracts only the three required Female / Neutral / Male SMPL model files.



### v6 — ZIP-recursive multi-source update

New in v6:

- ZIPs are indexed without full extraction.
- Videos are discovered through arbitrarily deep nested folders.
- Only one ZIP-contained video is temporarily extracted at a time.
- Temporary ZIP videos are removed after processing.
- Each group can use multiple source ZIPs/folders.
- High-school supports two or more sources.
- Test mode selects 3 videos per **group**, round-robin across sources.
- `input_video_manifest.csv` maps every internal ID back to the original ZIP/path/file.

### MediaPipe Tasks fix

- Removed runtime dependency on `mp.solutions`.
- Downloads Google's official `pose_landmarker_full.task` automatically.
- Uses VIDEO mode and `detect_for_video()` with monotonically increasing timestamps.
- Retains the original 23 LightGBM input equations via a Tasks landmark adapter.
- QC skeletons are drawn directly with OpenCV.


In [ ]:

# ============================================================
# 0. Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ============================================================
# 1. USER SETTINGS — v12
# ============================================================
from pathlib import Path

GROUP_SOURCES = {
    "elementary": [
        Path("/content/drive/MyDrive/臨床研究_発表/投球動作解析/投球動画　Full/2024小学校検診動画.zip"),
    ],
    "mixed": [
        Path("/content/drive/MyDrive/臨床研究_発表/投球動作解析/投球動画　Full/2026小中学生検診動画.zip"),
    ],
    "high": [
        Path("/content/drive/MyDrive/臨床研究_発表/投球動作解析/投球動画　Full/2025高校検診動画.zip"),
        Path("/content/drive/MyDrive/臨床研究_発表/投球動作解析/投球動画_高校生5phase/2024高校野球肘検診/元の動画/結合済み_選手ID別"),
    ],
}

GROUP_LABELS = {
    "elementary": "Elementary",
    "mixed": "Elementary + Junior high",
    "high": "High school",
}

GROUP_ORDER = {
    "elementary": 0,
    "mixed": 1,
    "high": 2,
}

OUTPUT_FOLDER = Path("/content/drive/MyDrive/OpenCapMono_Pitching_Output")
ASSET_FOLDER  = Path("/content/drive/MyDrive/OpenCapAssets")
RESEARCH_FOLDER = Path("/content/drive/MyDrive/臨床研究_発表/投球動作解析")

TEST_MODE = True
TEST_N_PER_SOURCE = 1
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".m4v"}

PHASE_MODEL_FILENAME = "throwdetect_trained_lgb_model_20250108_5phase.pkl"
PHASE_MODEL_PRIMARY_PATH = Path("/content/drive/MyDrive/臨床研究_発表/投球動作解析/throwdetect_trained_lgb_model_20250108_5phase.pkl")
PHASE_MODEL_HINTS = [
    PHASE_MODEL_PRIMARY_PATH,
    RESEARCH_FOLDER / PHASE_MODEL_FILENAME,
    ASSET_FOLDER / PHASE_MODEL_FILENAME,
    Path("/content/drive/MyDrive") / PHASE_MODEL_FILENAME,
]
PHASE_MODEL_PATH = None

# MediaPipe speed settings
MEDIAPIPE_MODEL_VARIANT = "full"   # "lite", "full", "heavy"
TARGET_MEDIAPIPE_FPS = 30.0        # 60fps -> stride2, 120fps -> stride4
MEDIAPIPE_MAX_DIM = 960            # inference image downscale only; 0=no resize
MEDIAPIPE_DELEGATE = "CPU"         # CPU recommended for stability
GENERATE_QC_VIDEO = True if TEST_MODE else False

# OpenCap rerun behavior
# False = reuse completed WHAM output when wham_output.pkl already exists.
# True  = force WHAM to rerun from scratch.
FORCE_RERUN_OPENCAP = False

PHASE_MEDIAN_WINDOW = 5
ENFORCE_MONOTONIC_PHASE = True

SMPL_ZIP_HINTS = [
    Path("/content/drive/SMPL_python_v.1.1.0.zip"),
    Path("/content/drive/MyDrive/SMPL_python_v.1.1.0.zip"),
    ASSET_FOLDER / "SMPL_python_v.1.1.0.zip",
]
SMPL_ZIP_PATH = SMPL_ZIP_HINTS[0]

REPO_DIR = Path("/content/opencap-monocular")

DEFAULT_HEIGHT_M = 1.60
DEFAULT_MASS_KG = 50.0
DEFAULT_SEX = "male"
DEFAULT_THROWING_SIDE = "R"

USER_CALIB_PATH = None
USER_INTRINSICS_PATH = None

ZIP_VIDEO_CACHE = Path("/content/pitching_zip_video_cache")
ZIP_VIDEO_CACHE.mkdir(parents=True, exist_ok=True)

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
ASSET_FOLDER.mkdir(parents=True, exist_ok=True)

print("TEST_MODE:", TEST_MODE)
for group, sources in GROUP_SOURCES.items():
    print(f"{group:12s}:")
    for j, source in enumerate(sources, start=1):
        print(f"  source {j}: {source}")

print("OUTPUT:", OUTPUT_FOLDER)
print("PHASE MODEL primary:", PHASE_MODEL_PRIMARY_PATH)
print("SMPL ZIP hint:", SMPL_ZIP_PATH)
print("MediaPipe:", MEDIAPIPE_MODEL_VARIANT, f"target={TARGET_MEDIAPIPE_FPS} Hz", f"max_dim={MEDIAPIPE_MAX_DIM}")
print("QC video:", GENERATE_QC_VIDEO)
print("TEST videos/source:", TEST_N_PER_SOURCE)


In [ ]:
# ============================================================
# 1A. Phase-model exact-path check
# ============================================================
print("Expected phase-model path:", PHASE_MODEL_PRIMARY_PATH)
if PHASE_MODEL_PRIMARY_PATH.exists():
    print("PKL exact path: FOUND")
else:
    print("PKL exact path: not found; section 4A will use bounded fallback search")


### v11 — GPU preflight added

OpenCap/WHAM requires an NVIDIA GPU in this Colab workflow. v11 checks the runtime **before** the heavy OpenCap installation.

If the check fails, in Colab choose **Runtime → Change runtime type → Hardware accelerator → T4 GPU** (or another NVIDIA GPU), reconnect, and run from the top.

`DPVO is not properly installed` is expected in the Slim/static-camera workflow. `apex is not installed` messages are also non-fatal for this pipeline.


In [ ]:
# ============================================================
# 1B. COLAB GPU PREFLIGHT — run before OpenCap installation
# ============================================================
import os
import subprocess
import sys

def require_colab_nvidia_gpu():
    print("Checking NVIDIA GPU availability...")

    # Host-level check first
    smi = subprocess.run(
        ["bash", "-lc", "command -v nvidia-smi >/dev/null 2>&1 && nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader || true"],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    gpu_lines = [x.strip() for x in smi.stdout.splitlines() if x.strip()]

    if gpu_lines:
        print("nvidia-smi:")
        for line in gpu_lines:
            print("  ", line)
    else:
        print("nvidia-smi: no NVIDIA GPU visible")

    # Host Python torch may or may not already be installed; this is only diagnostic.
    host_torch_ok = False
    try:
        import torch
        host_torch_ok = bool(torch.cuda.is_available())
        print("Host torch:", torch.__version__)
        print("Host torch.cuda.is_available():", host_torch_ok)
        if host_torch_ok:
            print("Host CUDA device:", torch.cuda.get_device_name(0))
    except Exception as e:
        print("Host torch CUDA diagnostic unavailable:", repr(e))

    if not gpu_lines:
        raise RuntimeError(
            "\nNo NVIDIA GPU is attached to this Colab runtime.\n\n"
            "OpenCap Monocular / WHAM cannot proceed in this notebook without CUDA.\n"
            "In Google Colab choose:\n"
            "  Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU\n"
            "(A100/L4 also work if available), then reconnect and run the notebook from the top.\n\n"
            "Installing a CUDA-enabled PyTorch wheel is NOT enough by itself; "
            "the Colab host must actually provide an NVIDIA GPU and driver."
        )

    print("\nGPU preflight: OK")
    return True

GPU_RUNTIME_OK = require_colab_nvidia_gpu()



## v6 input design — ZIP / folder / nested folders

`GROUP_SOURCES` accepts **multiple ZIP files and/or normal folders per group**.

The current defaults point to:

- 2024 elementary-school screening ZIP
- 2026 mixed elementary/junior-high screening ZIP
- 2025 high-school screening ZIP
- one additional high-school source slot

### Recursive screening

For every source, this notebook:

1. identifies ZIP vs normal folder,
2. recursively screens **all nested levels**,
3. finds `.mp4 / .mov / .avi / .m4v`,
4. does **not** fully extract large ZIP archives,
5. extracts only the currently processed video to Colab local storage,
6. removes that temporary ZIP video after each MediaPipe/OpenCap step,
7. assigns a short unique internal video ID,
8. saves the original ZIP/path/filename mapping to `input_video_manifest.csv`.

### Test mode

```python
TEST_MODE = True
TEST_N_PER_SOURCE = 1
```

selects **1 video from each configured input source** (each ZIP or folder).  
With the current four sources, the test run therefore uses up to **4 videos total**.

For the final run:

```python
TEST_MODE = False
```


In [ ]:
# ============================================================
# 2. Host-side dependencies — v14 Colab/Python 3.13 safe install
# ============================================================
import sys
import subprocess

print("Host Python:", sys.version)
print("Executable:", sys.executable)

def pip_install(args, label):
    cmd = [sys.executable, "-m", "pip", "install"] + list(args)
    print("\n[" + label + "]")
    print(">", " ".join(cmd))
    proc = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(proc.stdout[-12000:])
    if proc.returncode != 0:
        raise RuntimeError(
            f"{label} failed with returncode={proc.returncode}\n"
            f"{proc.stdout[-12000:]}"
        )

# Keep packaging tools current first.
pip_install(
    ["-q", "--upgrade", "pip", "setuptools", "wheel"],
    "packaging tools"
)

# IMPORTANT:
# Install MediaPipe separately and require a prebuilt wheel.
# mediapipe 0.10.35 provides a Linux py3 wheel, including current Colab.
pip_install(
    [
        "-q",
        "--only-binary=:all:",
        "mediapipe==0.10.35",
    ],
    "MediaPipe"
)

# Verify immediately, before installing anything else.
import mediapipe as mp
print("MediaPipe import: OK")
print("MediaPipe version:", mp.__version__)

# Remaining host packages.
# DO NOT pin scikit-learn to 1.4.2 on Python 3.13:
# that release has no cp313 wheel and attempts a source build.
pip_install(
    [
        "-q",
        "--prefer-binary",
        "opencv-python-headless<5",
        "lightgbm>=4.0",
        "scikit-learn>=1.6",
        "scipy>=1.10",
        "pandas>=2.0",
        "matplotlib>=3.7",
    ],
    "analysis dependencies"
)

# Imports after installation
import os
import cv2
import math
import pickle
import shutil
import zipfile
import warnings
import urllib.request
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mediapipe as mp
import sklearn
import lightgbm

from scipy.ndimage import median_filter
from scipy.signal import savgol_filter
from scipy.stats import linregress
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("\nPackage versions")
print("Python      :", sys.version.split()[0])
print("OpenCV      :", cv2.__version__)
print("MediaPipe   :", mp.__version__)
print("NumPy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("LightGBM    :", lightgbm.__version__)

POSE_MODEL_URLS = {
    "lite": (
        "https://storage.googleapis.com/mediapipe-models/"
        "pose_landmarker/pose_landmarker_lite/float16/latest/"
        "pose_landmarker_lite.task"
    ),
    "full": (
        "https://storage.googleapis.com/mediapipe-models/"
        "pose_landmarker/pose_landmarker_full/float16/latest/"
        "pose_landmarker_full.task"
    ),
    "heavy": (
        "https://storage.googleapis.com/mediapipe-models/"
        "pose_landmarker/pose_landmarker_heavy/float16/latest/"
        "pose_landmarker_heavy.task"
    ),
}

if MEDIAPIPE_MODEL_VARIANT not in POSE_MODEL_URLS:
    raise ValueError("MEDIAPIPE_MODEL_VARIANT must be 'lite', 'full', or 'heavy'.")

POSE_TASK_MODEL = Path(f"/content/pose_landmarker_{MEDIAPIPE_MODEL_VARIANT}.task")
POSE_TASK_URL = POSE_MODEL_URLS[MEDIAPIPE_MODEL_VARIANT]

if (not POSE_TASK_MODEL.exists()) or POSE_TASK_MODEL.stat().st_size < 1_000_000:
    print(f"Downloading MediaPipe Pose Landmarker {MEDIAPIPE_MODEL_VARIANT}...")
    urllib.request.urlretrieve(POSE_TASK_URL, POSE_TASK_MODEL)

print(
    "Pose task model:",
    POSE_TASK_MODEL,
    f"({POSE_TASK_MODEL.stat().st_size / 1024 / 1024:.1f} MB)"
)

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

def create_pose_landmarker():
    delegate_name = str(MEDIAPIPE_DELEGATE).upper()

    if delegate_name == "GPU":
        try:
            base = BaseOptions(
                model_asset_path=str(POSE_TASK_MODEL),
                delegate=BaseOptions.Delegate.GPU,
            )
        except Exception as e:
            print("MediaPipe GPU delegate setup failed; falling back to CPU:", repr(e))
            base = BaseOptions(
                model_asset_path=str(POSE_TASK_MODEL),
                delegate=BaseOptions.Delegate.CPU,
            )
    else:
        base = BaseOptions(
            model_asset_path=str(POSE_TASK_MODEL),
            delegate=BaseOptions.Delegate.CPU,
        )

    options = PoseLandmarkerOptions(
        base_options=base,
        running_mode=VisionRunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_segmentation_masks=False,
    )
    return PoseLandmarker.create_from_options(options)

class LandmarkListAdapter:
    def __init__(self, landmarks):
        self.landmark = landmarks

POSE_CONNECTIONS_TASKS = [
    (0,1),(1,2),(2,3),(3,7),
    (0,4),(4,5),(5,6),(6,8),
    (9,10),
    (11,12),
    (11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
    (11,23),(12,24),(23,24),
    (23,25),(25,27),(27,29),(29,31),(27,31),
    (24,26),(26,28),(28,30),(30,32),(28,32),
]

def _landmark_visible(lm, threshold=0.15):
    vis = getattr(lm, "visibility", None)
    pres = getattr(lm, "presence", None)
    if vis is not None and vis < threshold:
        return False
    if pres is not None and pres < threshold:
        return False
    return True

def draw_tasks_pose(frame, landmarks, color=(255,255,255),
                    line_thickness=3, radius=3):
    if landmarks is None or len(landmarks) < 33:
        return frame

    h, w = frame.shape[:2]

    def pt(i):
        lm = landmarks[i]
        return (
            int(np.clip(lm.x, 0.0, 1.0) * (w - 1)),
            int(np.clip(lm.y, 0.0, 1.0) * (h - 1)),
        )

    for a, b in POSE_CONNECTIONS_TASKS:
        if _landmark_visible(landmarks[a]) and _landmark_visible(landmarks[b]):
            cv2.line(frame, pt(a), pt(b), color, line_thickness, cv2.LINE_AA)

    for lm in landmarks:
        if _landmark_visible(lm):
            x = int(np.clip(lm.x, 0.0, 1.0) * (w - 1))
            y = int(np.clip(lm.y, 0.0, 1.0) * (h - 1))
            cv2.circle(frame, (x,y), radius, color, -1, cv2.LINE_AA)

    return frame

print("\nMediaPipe Tasks Pose Landmarker: READY")


In [ ]:
# ============================================================
# 3. Resolve ZIP/folder sources and recursively scan videos — v15
#    TEST_MODE=True => 1 video per configured source
# ============================================================
import hashlib
import os
import shutil
import zipfile
import pandas as pd
import numpy as np

def _short_hash(text, n=12):
    return hashlib.sha1(str(text).encode("utf-8")).hexdigest()[:n]

def _windows_drive_to_colab(path_like):
    """Allow pasted Google Drive for desktop paths such as G:\\マイドライブ\\..."""
    raw = str(path_like)
    norm = raw.replace("\\", "/")
    prefixes = [
        "G:/マイドライブ/",
        "G:/My Drive/",
        "G:/MyDrive/",
    ]
    for prefix in prefixes:
        if norm.lower().startswith(prefix.lower()):
            tail = norm[len(prefix):]
            return Path("/content/drive/MyDrive") / Path(tail)
    return Path(raw)

def _resolve_source_path(source):
    source = _windows_drive_to_colab(source)
    if source.exists():
        return source

    # Bounded exact-name fallback inside the research folder.
    # This is intentionally not an unrestricted search of all Google Drive.
    target_name = source.name
    if RESEARCH_FOLDER.exists() and target_name:
        try:
            hits = [
                p for p in RESEARCH_FOLDER.rglob(target_name)
                if p.name == target_name
            ]
        except Exception:
            hits = []

        if hits:
            hits = sorted(hits, key=lambda p: (len(p.parts), str(p)))
            print(f"Source auto-resolved: {source} -> {hits[0]}")
            return hits[0]

    return source

def _scan_zip(zip_path, group_key, source_index):
    records = []
    zip_path = Path(zip_path)

    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            for info in zf.infolist():
                if info.is_dir():
                    continue

                member = info.filename
                if "__MACOSX" in member:
                    continue
                if Path(member).name.startswith("._"):
                    continue
                if Path(member).suffix.lower() not in VIDEO_EXTENSIONS:
                    continue

                suffix = Path(member).suffix.lower()
                analysis_id = (
                    f"{group_key}"
                    f"__s{source_index:02d}"
                    f"__{_short_hash(str(zip_path) + '|' + member)}"
                    f"{suffix}"
                )
                original_name = Path(member).name

                records.append({
                    "filename": analysis_id,
                    "original_filename": original_name,
                    "original_stem": Path(original_name).stem,
                    "video_path": None,
                    "source_group": group_key,
                    "source_index": source_index,
                    "source_type": "zip",
                    "source_container": str(zip_path),
                    "source_relpath": member,
                    "zip_member": member,
                    "subject_id": Path(original_name).stem,
                    "age": np.nan,
                    "sex": DEFAULT_SEX,
                    "throwing_side": DEFAULT_THROWING_SIDE,
                    "group": group_key,
                    "group_label": GROUP_LABELS[group_key],
                    "group_order": GROUP_ORDER[group_key],
                })
    except zipfile.BadZipFile as e:
        print(f"WARNING [{group_key}] bad ZIP: {zip_path} -> {e}")

    return records

def _scan_folder(folder_path, group_key, source_index):
    records = []
    folder_path = Path(folder_path)

    # recursive scan through player-ID subfolders as well
    for p in folder_path.rglob("*"):
        if not p.is_file():
            continue
        if p.name.startswith("._"):
            continue
        if "__MACOSX" in p.parts:
            continue
        if p.suffix.lower() not in VIDEO_EXTENSIONS:
            continue

        try:
            rel = p.relative_to(folder_path).as_posix()
        except Exception:
            rel = p.name

        analysis_id = (
            f"{group_key}"
            f"__s{source_index:02d}"
            f"__{_short_hash(str(folder_path) + '|' + rel)}"
            f"{p.suffix.lower()}"
        )

        # For the player-ID folder structure, use first relative folder as
        # subject_id when available; otherwise fall back to filename stem.
        rel_parts = Path(rel).parts
        subject_id = rel_parts[0] if len(rel_parts) > 1 else p.stem

        records.append({
            "filename": analysis_id,
            "original_filename": p.name,
            "original_stem": p.stem,
            "video_path": str(p),
            "source_group": group_key,
            "source_index": source_index,
            "source_type": "folder",
            "source_container": str(folder_path),
            "source_relpath": rel,
            "zip_member": None,
            "subject_id": subject_id,
            "age": np.nan,
            "sex": DEFAULT_SEX,
            "throwing_side": DEFAULT_THROWING_SIDE,
            "group": group_key,
            "group_label": GROUP_LABELS[group_key],
            "group_order": GROUP_ORDER[group_key],
        })

    return records

def _select_test_from_source(records, n=1):
    records = sorted(
        records,
        key=lambda x: (
            str(x["source_relpath"]).lower(),
            str(x["original_filename"]).lower(),
        )
    )
    return records[:max(0, int(n))]

all_records = []
source_scan_rows = []

for group_key, sources in GROUP_SOURCES.items():
    for source_index, source in enumerate(sources, start=1):
        if source is None:
            print(f"[{group_key}] source {source_index}: not specified -> skip")
            continue

        source = _resolve_source_path(source)

        if not source.exists():
            print(f"WARNING [{group_key}] source not found: {source}")
            source_scan_rows.append({
                "group": group_key,
                "source_index": source_index,
                "source": str(source),
                "status": "not_found",
                "n_found": 0,
                "n_selected": 0,
            })
            continue

        if source.is_file() and source.suffix.lower() == ".zip":
            records = _scan_zip(source, group_key, source_index)
            source_type = "zip"
        elif source.is_dir():
            records = _scan_folder(source, group_key, source_index)
            source_type = "folder"
        else:
            print(f"WARNING [{group_key}] unsupported source: {source}")
            continue

        n_found = len(records)

        if TEST_MODE:
            selected = _select_test_from_source(records, TEST_N_PER_SOURCE)
        else:
            selected = sorted(
                records,
                key=lambda x: str(x["source_relpath"]).lower()
            )

        print(
            f"[{group_key}] source {source_index} ({source_type}): "
            f"{n_found} videos found -> {len(selected)} selected"
        )

        if source_type == "folder" and n_found:
            # Show a few paths so nested-folder discovery can be checked immediately.
            for ex in records[:3]:
                print("   found:", ex["source_relpath"])

        source_scan_rows.append({
            "group": group_key,
            "source_index": source_index,
            "source": str(source),
            "source_type": source_type,
            "status": "ok",
            "n_found": n_found,
            "n_selected": len(selected),
        })

        all_records.extend(selected)

source_scan_df = pd.DataFrame(source_scan_rows)
display(source_scan_df)

if not all_records:
    raise RuntimeError(
        "No videos found. Check GROUP_SOURCES and the source_scan_df table above."
    )

video_table = pd.DataFrame(all_records).reset_index(drop=True)

manifest_cols = [
    "filename",
    "original_filename",
    "original_stem",
    "group",
    "group_label",
    "source_index",
    "source_type",
    "source_container",
    "source_relpath",
    "zip_member",
    "subject_id",
]

video_table[manifest_cols].to_csv(
    OUTPUT_FOLDER / "input_video_manifest.csv",
    index=False
)
source_scan_df.to_csv(
    OUTPUT_FOLDER / "input_source_scan.csv",
    index=False
)

display(
    video_table[
        [
            "group_label",
            "source_index",
            "source_type",
            "original_filename",
            "source_relpath",
            "filename",
        ]
    ]
)

print("\nN selected by source:")
display(
    video_table.groupby(
        ["group_label", "source_index", "source_container"]
    )
    .size()
    .rename("N")
    .reset_index()
)

print("\nTOTAL selected N =", len(video_table))


In [ ]:

# ============================================================
# 3B. Materialize one video only when needed
#     ZIP: extract one file -> analyze -> delete
#     Folder: use original file directly
# ============================================================
def materialize_video(row):
    source_type = str(row["source_type"])

    if source_type == "folder":
        p = Path(row["video_path"])
        if not p.exists():
            raise FileNotFoundError(p)
        return p

    if source_type == "zip":
        zip_path = Path(row["source_container"])
        member = row["zip_member"]

        dest_dir = ZIP_VIDEO_CACHE / row["group"]
        dest_dir.mkdir(parents=True, exist_ok=True)

        dest = dest_dir / row["filename"]

        if dest.exists() and dest.stat().st_size > 0:
            return dest

        with zipfile.ZipFile(zip_path, "r") as zf:
            with zf.open(member, "r") as src, open(dest, "wb") as dst:
                shutil.copyfileobj(src, dst)

        return dest

    raise RuntimeError(f"Unknown source_type: {source_type}")

def cleanup_materialized_video(row, local_path):
    # Never delete source files from a normal folder / Google Drive.
    if str(row["source_type"]) != "zip":
        return

    p = Path(local_path)

    try:
        if p.exists():
            p.unlink()
    except Exception as e:
        print("Temporary-file cleanup warning:", e)



# 4. Existing 5-phase LightGBM model

The original program uses MediaPipe `pose_world_landmarks` and a fixed feature vector.

This notebook reproduces the same **23 model inputs** in the same order.

The uploaded classifier uses labels **1–5**, so this notebook maps:

| Model class | phase_id | phase |
|---:|---:|---|
| 1 | 0 | Windup |
| 2 | 1 | Stride |
| 3 | 2 | ArmCocking |
| 4 | 3 | ArmAcceleration |
| 5 | 4 | FollowThrough |

The original prediction is retained as `model_pred_raw`.  
The mapped phase is retained as `phase_raw`, and a temporal-cleaned version as `phase_clean`.


In [ ]:
# ============================================================
# 4A. Auto-detect and load the existing 5-phase model
# ============================================================
from pathlib import Path

def resolve_phase_model_path():
    # Windows Google Drive "G:\\マイドライブ\\..." maps to
    # Colab "/content/drive/MyDrive/..." after drive.mount().
    if PHASE_MODEL_PRIMARY_PATH.exists():
        print("Using exact phase-model path:", PHASE_MODEL_PRIMARY_PATH)
        return PHASE_MODEL_PRIMARY_PATH

    for p in PHASE_MODEL_HINTS:
        p = Path(p)
        if p.exists() and p.is_file():
            return p

    roots = [RESEARCH_FOLDER, ASSET_FOLDER]
    candidates = []

    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        try:
            candidates.extend(root.rglob(PHASE_MODEL_FILENAME))
        except Exception as e:
            print("Phase model search warning:", root, repr(e))

    candidates = sorted(
        {p.resolve() for p in candidates if p.is_file()},
        key=lambda p: (len(p.parts), str(p))
    )

    if candidates:
        return candidates[0]

    raise FileNotFoundError(
        "Could not find the phase model. Checked:\n"
        + "\n".join(str(p) for p in PHASE_MODEL_HINTS)
        + f"\nand bounded searches below {RESEARCH_FOLDER} and {ASSET_FOLDER}"
    )

from sklearn.exceptions import InconsistentVersionWarning

# The model was serialized with sklearn 1.4.2.
# Current Colab may use a newer sklearn because Python 3.13 cannot install
# sklearn 1.4.2 from a binary wheel. v12 already demonstrated successful
# loading/inference. We validate feature count/names/classes below.
warnings.filterwarnings("ignore", category=InconsistentVersionWarning)

PHASE_MODEL_PATH = resolve_phase_model_path()
print("Phase model detected:", PHASE_MODEL_PATH)

with open(PHASE_MODEL_PATH, "rb") as f:
    phase_model = pickle.load(f)

print("Model type:", type(phase_model))
print("n_features_in_:", getattr(phase_model, "n_features_in_", None))
print("classes_:", getattr(phase_model, "classes_", None))
print("feature_name_:", getattr(phase_model, "feature_name_", None))

EXPECTED_PHASE_FEATURES = [
    "norm_rt_forearm_dist",
    "norm_rt_uparm_dist",
    "norm_rt_hip_dist",
    "norm_rt_knee_dist",
    "norm_lt_hip_dist",
    "norm_lt_knee_dist",
    "rt_elbow_angle",
    "rt_shoulder_angle",
    "lt_elbow_angle",
    "lt_shoulder_angle",
    "rt_hip_angle",
    "rt_knee_angle",
    "lt_hip_angle",
    "lt_knee_angle",
    "shoulder_hip_ratio",
    "norm_rt_elbow_size",
    "norm_rt_shoulder_size",
    "norm_rt_trunk_size",
    "norm_lt_trunk_size",
    "norm_rt_hip_size",
    "norm_rt_knee_size",
    "norm_lt_hip_size",
    "norm_lt_knee_size",
]

if getattr(phase_model, "n_features_in_", len(EXPECTED_PHASE_FEATURES)) != len(EXPECTED_PHASE_FEATURES):
    raise ValueError("Phase model feature count does not match the original 23-feature pipeline.")

model_feature_names = getattr(phase_model, "feature_name_", None)
if model_feature_names is not None and list(model_feature_names) != EXPECTED_PHASE_FEATURES:
    raise ValueError("Phase model feature names/order differ from expected original 23 features.")

print("Phase model feature compatibility: OK")


In [ ]:

# ============================================================
# 4B. Original-feature-compatible geometry functions
# ============================================================
def calculate_distance(x1, y1, x2, y2):
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

def calc_angle(x0, y0, x1, y1, x2, y2):
    vec1 = np.array([x1 - x0, y1 - y0], float)
    vec2 = np.array([x2 - x0, y2 - y0], float)
    n1 = np.linalg.norm(vec1)
    n2 = np.linalg.norm(vec2)
    if n1 == 0 or n2 == 0:
        return 0.0
    c = float(np.clip(np.dot(vec1, vec2) / (n1 * n2), -1.0, 1.0))
    return math.degrees(math.acos(c))

def calc_outerproduct(x0, y0, x1, y1, x2, y2):
    # 2D scalar cross product (determinant).
    # Mathematically identical to np.cross([x1,y1],[x2,y2]) in old NumPy,
    # but avoids the NumPy 2.x deprecation warning for 2D vectors.
    ax, ay = (x1 - x0), (y1 - y0)
    bx, by = (x2 - x0), (y2 - y0)
    return float((ax * by - ay * bx) * 1000.0)

def calc_angle_3d(p0, p1, p2):
    v1 = np.asarray(p1, float) - np.asarray(p0, float)
    v2 = np.asarray(p2, float) - np.asarray(p0, float)
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        return 0.0
    c = float(np.clip(np.dot(v1, v2) / (n1*n2), -1.0, 1.0))
    return math.degrees(math.acos(c))

def _xyz(world_landmarks, idx):
    p = world_landmarks.landmark[idx]
    return np.array([p.x, p.y, p.z], float)

def _xy(world_landmarks, idx):
    p = world_landmarks.landmark[idx]
    return float(p.x), float(p.y)


In [ ]:

# ============================================================
# 4C. Build the exact 23 phase-classifier inputs
# ============================================================
def compute_original_phase_features(world_landmarks):
    # MediaPipe landmark indices used by the original program:
    # shoulders 11/12, elbows 13/14, wrists 15/16,
    # hips 23/24, knees 25/26, ankles 27/28.

    rs = world_landmarks.landmark[12]
    re = world_landmarks.landmark[14]
    rw = world_landmarks.landmark[16]

    ls = world_landmarks.landmark[11]
    le = world_landmarks.landmark[13]
    lw = world_landmarks.landmark[15]

    rh = world_landmarks.landmark[24]
    rk = world_landmarks.landmark[26]
    ra = world_landmarks.landmark[28]

    lh = world_landmarks.landmark[23]
    lk = world_landmarks.landmark[25]
    la = world_landmarks.landmark[27]

    # Same x-y distances as original code
    rt_forearm_dist = calculate_distance(rw.x, rw.y, re.x, re.y)
    rt_trunk_dist   = calculate_distance(rs.x, rs.y, rh.x, rh.y)
    rt_uparm_dist   = calculate_distance(rs.x, rs.y, re.x, re.y)
    rt_hip_dist     = calculate_distance(rh.x, rh.y, rk.x, rk.y)
    rt_knee_dist    = calculate_distance(rk.x, rk.y, ra.x, ra.y)
    lt_hip_dist     = calculate_distance(lh.x, lh.y, lk.x, lk.y)
    lt_knee_dist    = calculate_distance(lk.x, lk.y, la.x, la.y)

    if rt_trunk_dist == 0:
        rt_trunk_dist = 1e-6

    norm_rt_forearm_dist = rt_forearm_dist / rt_trunk_dist
    norm_rt_uparm_dist   = rt_uparm_dist / rt_trunk_dist
    norm_rt_hip_dist     = rt_hip_dist / rt_trunk_dist
    norm_rt_knee_dist    = rt_knee_dist / rt_trunk_dist
    norm_lt_hip_dist     = lt_hip_dist / rt_trunk_dist
    norm_lt_knee_dist    = lt_knee_dist / rt_trunk_dist

    shoulder_dist = calculate_distance(rs.x, rs.y, ls.x, ls.y)
    hip_dist      = calculate_distance(rh.x, rh.y, lh.x, lh.y)
    shoulder_hip_ratio = shoulder_dist / hip_dist if hip_dist != 0 else 0.0

    rt_elbow_angle    = calc_angle(re.x, re.y, rs.x, rs.y, rw.x, rw.y)
    rt_shoulder_angle = calc_angle(rs.x, rs.y, rh.x, rh.y, re.x, re.y)
    lt_elbow_angle    = calc_angle(le.x, le.y, ls.x, ls.y, lw.x, lw.y)
    lt_shoulder_angle = calc_angle(ls.x, ls.y, lh.x, lh.y, le.x, le.y)
    rt_hip_angle      = calc_angle(rh.x, rh.y, rs.x, rs.y, rk.x, rk.y)
    rt_knee_angle     = calc_angle(rk.x, rk.y, rh.x, rh.y, ra.x, ra.y)
    lt_hip_angle      = calc_angle(lh.x, lh.y, ls.x, ls.y, lk.x, lk.y)
    lt_knee_angle     = calc_angle(lk.x, lk.y, lh.x, lh.y, la.x, la.y)

    rt_elbow_size    = calc_outerproduct(re.x, re.y, rs.x, rs.y, rw.x, rw.y)
    rt_shoulder_size = calc_outerproduct(rs.x, rs.y, re.x, re.y, rh.x, rh.y)
    rt_trunk_size    = calc_outerproduct(rs.x, rs.y, ls.x, ls.y, rh.x, rh.y)
    lt_trunk_size    = calc_outerproduct(ls.x, ls.y, rs.x, rs.y, lh.x, lh.y)
    rt_hip_size      = calc_outerproduct(rh.x, rh.y, rk.x, rk.y, rs.x, rs.y)
    rt_knee_size     = calc_outerproduct(rk.x, rk.y, ra.x, ra.y, rh.x, rh.y)
    lt_hip_size      = calc_outerproduct(lh.x, lh.y, lk.x, lk.y, ls.x, ls.y)
    lt_knee_size     = calc_outerproduct(lk.x, lk.y, la.x, la.y, lh.x, lh.y)

    denom2 = rt_trunk_dist ** 2
    vals = {
        "norm_rt_forearm_dist": norm_rt_forearm_dist,
        "norm_rt_uparm_dist": norm_rt_uparm_dist,
        "norm_rt_hip_dist": norm_rt_hip_dist,
        "norm_rt_knee_dist": norm_rt_knee_dist,
        "norm_lt_hip_dist": norm_lt_hip_dist,
        "norm_lt_knee_dist": norm_lt_knee_dist,
        "rt_elbow_angle": rt_elbow_angle,
        "rt_shoulder_angle": rt_shoulder_angle,
        "lt_elbow_angle": lt_elbow_angle,
        "lt_shoulder_angle": lt_shoulder_angle,
        "rt_hip_angle": rt_hip_angle,
        "rt_knee_angle": rt_knee_angle,
        "lt_hip_angle": lt_hip_angle,
        "lt_knee_angle": lt_knee_angle,
        "shoulder_hip_ratio": shoulder_hip_ratio,
        "norm_rt_elbow_size": rt_elbow_size / denom2,
        "norm_rt_shoulder_size": rt_shoulder_size / denom2,
        "norm_rt_trunk_size": rt_trunk_size / denom2,
        "norm_lt_trunk_size": lt_trunk_size / denom2,
        "norm_rt_hip_size": rt_hip_size / denom2,
        "norm_rt_knee_size": rt_knee_size / denom2,
        "norm_lt_hip_size": lt_hip_size / denom2,
        "norm_lt_knee_size": lt_knee_size / denom2,
    }
    return vals


In [ ]:

# ============================================================
# 4D. Body-proportion features from MediaPipe world landmarks
# ============================================================
def d3(a, b):
    return float(np.linalg.norm(np.asarray(a) - np.asarray(b)))

def compute_morphology_features(world_landmarks):
    # Use 3D world landmarks only for scale-free segment ratios.
    LS = _xyz(world_landmarks, 11)
    RS = _xyz(world_landmarks, 12)
    LE = _xyz(world_landmarks, 13)
    RE = _xyz(world_landmarks, 14)
    LW = _xyz(world_landmarks, 15)
    RW = _xyz(world_landmarks, 16)

    LH = _xyz(world_landmarks, 23)
    RH = _xyz(world_landmarks, 24)
    LK = _xyz(world_landmarks, 25)
    RK = _xyz(world_landmarks, 26)
    LA = _xyz(world_landmarks, 27)
    RA = _xyz(world_landmarks, 28)

    LEAR = _xyz(world_landmarks, 7)
    REAR = _xyz(world_landmarks, 8)
    NOSE = _xyz(world_landmarks, 0)

    shoulder_mid = (LS + RS) / 2
    hip_mid = (LH + RH) / 2

    trunk = d3(shoulder_mid, hip_mid)
    thigh = np.median([d3(LH, LK), d3(RH, RK)])
    shank = np.median([d3(LK, LA), d3(RK, RA)])
    upperarm = np.median([d3(LS, LE), d3(RS, RE)])
    forearm = np.median([d3(LE, LW), d3(RE, RW)])

    leg = thigh + shank
    arm = upperarm + forearm

    shoulder_width = d3(LS, RS)
    pelvis_width = d3(LH, RH)
    ear_mid = (LEAR + REAR) / 2
    head_proxy = d3(ear_mid, NOSE)

    e = 1e-8
    return {
        "leg_trunk_ratio": leg / (trunk + e),
        "thigh_trunk_ratio": thigh / (trunk + e),
        "shank_trunk_ratio": shank / (trunk + e),
        "arm_trunk_ratio": arm / (trunk + e),
        "forearm_upperarm_ratio": forearm / (upperarm + e),
        "shoulder_pelvis_width_ratio": shoulder_width / (pelvis_width + e),
        "head_trunk_ratio_proxy": head_proxy / (trunk + e),  # secondary only
    }

MORPH_PRIMARY_FEATURES = [
    "leg_trunk_ratio",
    "thigh_trunk_ratio",
    "shank_trunk_ratio",
    "arm_trunk_ratio",
    "forearm_upperarm_ratio",
    "shoulder_pelvis_width_ratio",
]


In [ ]:

# ============================================================
# 4E. Phase mapping + temporal cleaning
# ============================================================
PHASE_NAMES = {
    0: "Windup",
    1: "Stride",
    2: "ArmCocking",
    3: "ArmAcceleration",
    4: "FollowThrough",
}

def prediction_to_phase_id(pred, model):
    # Current uploaded LGBMClassifier has classes 1..5.
    classes = getattr(model, "classes_", None)

    if classes is not None:
        classes = np.asarray(classes)
        if np.array_equal(classes, np.array([1,2,3,4,5])):
            return int(np.clip(int(pred) - 1, 0, 4))
        if np.array_equal(classes, np.array([0,1,2,3,4])):
            return int(np.clip(int(pred), 0, 4))

    # Backward compatibility with a regression-style 0..4 model
    return int(np.clip(round(float(pred)), 0, 4))

def clean_phase_sequence(df, median_window=5, enforce_monotonic=True):
    out = df.copy()
    s = out["phase_raw"].astype(float)

    # interpolate only inside detected portions; then edge-fill
    s = s.interpolate(limit_direction="both")

    if median_window and median_window > 1 and len(s) >= median_window:
        if median_window % 2 == 0:
            median_window += 1
        s2 = median_filter(s.to_numpy(), size=median_window, mode="nearest")
    else:
        s2 = s.to_numpy()

    s2 = np.rint(s2).astype(int)
    s2 = np.clip(s2, 0, 4)

    if enforce_monotonic:
        # Pitching phases should not reverse in time.
        # Preserve skips, but prohibit backward phase transitions.
        s2 = np.maximum.accumulate(s2)

    out["phase_clean"] = s2
    out["phase_name"] = out["phase_clean"].map(PHASE_NAMES)
    return out



# 5. MediaPipe batch analysis

For each selected video this stage:

- recreates the original 23 LightGBM inputs,
- predicts 5 pitching phases,
- saves raw and cleaned phases,
- extracts scale-free morphology ratios,
- writes a QC video with phase overlay.

This stage is independent from OpenCap, so you can verify the existing phase model first.


In [ ]:
# ============================================================
# 5A. Analyze one video — FAST MediaPipe Tasks
# ============================================================
PHASE_QC_DIR = OUTPUT_FOLDER / "qc_phase_videos"
PHASE_QC_DIR.mkdir(parents=True, exist_ok=True)

def _resize_for_mediapipe(frame, max_dim):
    if not max_dim or max_dim <= 0:
        return frame
    h, w = frame.shape[:2]
    m = max(h, w)
    if m <= max_dim:
        return frame
    scale = float(max_dim) / float(m)
    nw = max(2, int(round(w * scale)))
    nh = max(2, int(round(h * scale)))
    return cv2.resize(frame, (nw, nh), interpolation=cv2.INTER_AREA)

def analyze_video_mediapipe(video_path, filename, group, group_label, subject_id):
    t0 = time.perf_counter()

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    if not np.isfinite(fps) or fps <= 0:
        fps = 30.0

    n_frames_meta = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n_frames_meta < 1:
        n_frames_meta = 0

    if TARGET_MEDIAPIPE_FPS and TARGET_MEDIAPIPE_FPS > 0:
        frame_stride = max(1, int(round(fps / float(TARGET_MEDIAPIPE_FPS))))
    else:
        frame_stride = 1

    effective_sampling_fps = fps / frame_stride

    sample_rows = []
    morph_rows = []
    feature_rows = []
    pose_cache = {}

    last_timestamp_ms = -1
    frame_idx = 0

    with create_pose_landmarker() as landmarker:
        while True:
            ok = cap.grab()
            if not ok:
                break

            if frame_idx % frame_stride == 0:
                ok, frame = cap.retrieve()
                if not ok:
                    frame_idx += 1
                    continue

                infer_frame = _resize_for_mediapipe(frame, MEDIAPIPE_MAX_DIM)
                rgb = cv2.cvtColor(infer_frame, cv2.COLOR_BGR2RGB)
                rgb = np.ascontiguousarray(rgb)

                mp_image = mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=rgb
                )

                timestamp_ms = int(round(frame_idx * 1000.0 / fps))
                if timestamp_ms <= last_timestamp_ms:
                    timestamp_ms = last_timestamp_ms + 1
                last_timestamp_ms = timestamp_ms

                result = landmarker.detect_for_video(mp_image, timestamp_ms)

                rec = {
                    "frame": frame_idx,
                    "detected": False,
                    "model_pred_raw": np.nan,
                    "phase_raw": np.nan,
                    "phase_confidence": np.nan,
                }

                has_pose = (
                    result is not None
                    and result.pose_world_landmarks is not None
                    and len(result.pose_world_landmarks) > 0
                    and result.pose_landmarks is not None
                    and len(result.pose_landmarks) > 0
                )

                if has_pose:
                    try:
                        pose_cache[frame_idx] = result.pose_landmarks[0]
                        world = LandmarkListAdapter(result.pose_world_landmarks[0])

                        feats = compute_original_phase_features(world)
                        feature_rows.append({"frame": frame_idx, **feats})
                        rec["detected"] = True

                        mf = compute_morphology_features(world)
                        mf.update({
                            "filename": filename,
                            "subject_id": subject_id,
                            "group": group,
                            "group_label": group_label,
                            "frame": frame_idx,
                            "time_s": frame_idx / fps,
                            "timestamp_ms": timestamp_ms,
                        })
                        morph_rows.append(mf)

                    except Exception as e:
                        rec["feature_error"] = repr(e)

                sample_rows.append(rec)

            frame_idx += 1

    cap.release()

    actual_n_frames = max(frame_idx, n_frames_meta)
    if actual_n_frames <= 0:
        raise RuntimeError(f"No frames decoded: {video_path}")

    sample_df = pd.DataFrame(sample_rows)

    # Batch LightGBM once per video
    if feature_rows:
        feat_df = pd.DataFrame(feature_rows).set_index("frame")
        X = feat_df[EXPECTED_PHASE_FEATURES]

        preds = phase_model.predict(X)
        if hasattr(phase_model, "predict_proba"):
            probs = phase_model.predict_proba(X)
            confs = np.nanmax(probs, axis=1)
        else:
            confs = np.full(len(X), np.nan)

        phase_ids = [prediction_to_phase_id(p, phase_model) for p in preds]

        pred_map = pd.DataFrame({
            "frame": X.index.to_numpy(),
            "model_pred_raw": np.asarray(preds, dtype=float),
            "phase_raw": np.asarray(phase_ids, dtype=float),
            "phase_confidence": np.asarray(confs, dtype=float),
        }).set_index("frame")

        sample_df = sample_df.set_index("frame")
        for col in ["model_pred_raw", "phase_raw", "phase_confidence"]:
            sample_df.loc[pred_map.index, col] = pred_map[col]
        sample_df = sample_df.reset_index()

        sample_df = sample_df.merge(
            feat_df.reset_index(),
            on="frame",
            how="left"
        )

    # Full original timeline
    phase_df = pd.DataFrame({
        "filename": filename,
        "subject_id": subject_id,
        "group": group,
        "group_label": group_label,
        "frame": np.arange(actual_n_frames, dtype=int),
    })
    phase_df["time_s"] = phase_df["frame"] / fps
    phase_df["timestamp_ms"] = np.rint(phase_df["time_s"] * 1000).astype(int)
    phase_df["fps"] = fps
    phase_df["n_frames"] = actual_n_frames
    phase_df["inference_stride"] = frame_stride
    phase_df["effective_inference_fps"] = effective_sampling_fps

    if not sample_df.empty:
        phase_df = phase_df.merge(sample_df, on="frame", how="left")
    else:
        phase_df["detected"] = False
        phase_df["model_pred_raw"] = np.nan
        phase_df["phase_raw"] = np.nan
        phase_df["phase_confidence"] = np.nan

    phase_df["is_inference_frame"] = phase_df["frame"] % frame_stride == 0

    valid_sample = phase_df["phase_raw"].notna()
    phase_df["phase_clean"] = np.nan
    phase_df["phase_name"] = None

    if valid_sample.any():
        sampled_for_clean = phase_df.loc[valid_sample].copy()

        clean_sample = clean_phase_sequence(
            sampled_for_clean,
            median_window=PHASE_MEDIAN_WINDOW,
            enforce_monotonic=ENFORCE_MONOTONIC_PHASE
        )

        sf = clean_sample["frame"].to_numpy(float)
        sp = clean_sample["phase_clean"].to_numpy(float)
        allf = phase_df["frame"].to_numpy(float)

        inside = (allf >= sf.min()) & (allf <= sf.max())
        interp = np.full(len(phase_df), np.nan)
        interp[inside] = np.interp(allf[inside], sf, sp)
        interp[inside] = np.clip(np.rint(interp[inside]), 0, 4)

        phase_df["phase_clean"] = interp
        phase_df.loc[inside, "phase_name"] = (
            phase_df.loc[inside, "phase_clean"].map(PHASE_NAMES)
        )

    morph_frame_df = pd.DataFrame(morph_rows)

    elapsed = time.perf_counter() - t0
    n_infer = len(sample_rows)
    n_pose = int(sample_df["detected"].fillna(False).sum()) if (not sample_df.empty and "detected" in sample_df) else 0

    perf = {
        "original_fps": fps,
        "n_video_frames": actual_n_frames,
        "frame_stride": frame_stride,
        "effective_inference_fps": effective_sampling_fps,
        "n_inference_frames": n_infer,
        "n_pose_frames": n_pose,
        "elapsed_mediapipe_s": elapsed,
        "inference_frames_per_sec": (n_infer / elapsed) if elapsed > 0 else np.nan,
        "video_seconds_per_wall_second": ((actual_n_frames / fps) / elapsed) if elapsed > 0 else np.nan,
        "mediapipe_model": MEDIAPIPE_MODEL_VARIANT,
        "mediapipe_max_dim": MEDIAPIPE_MAX_DIM,
    }

    print(
        f"MediaPipe FAST: video={actual_n_frames} frames @ {fps:.2f} fps | "
        f"inference={n_infer} frames @ ~{effective_sampling_fps:.1f} Hz | "
        f"stride={frame_stride} | elapsed={elapsed:.1f}s | "
        f"throughput={perf['inference_frames_per_sec']:.1f} infer-frames/s"
    )

    return phase_df, morph_frame_df, pose_cache, perf


In [ ]:
# ============================================================
# 5B. FAST QC renderer — no second MediaPipe inference
# ============================================================
PHASE_COLORS = {
    0: (220, 220, 220),
    1: (255, 200, 0),
    2: (0, 200, 255),
    3: (0, 100, 255),
    4: (180, 80, 255),
}

def render_phase_qc(video_path, phase_df, output_path, pose_cache=None):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not np.isfinite(fps) or fps <= 0:
        fps = 30.0

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h)
    )
    if not writer.isOpened():
        cap.release()
        raise RuntimeError(f"Could not open VideoWriter: {output_path}")

    phase_lookup = phase_df.set_index("frame")
    pose_cache = pose_cache or {}
    last_pose = None
    fi = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if fi in pose_cache:
            last_pose = pose_cache[fi]

        if fi in phase_lookup.index:
            rr = phase_lookup.loc[fi]
            ph = rr.get("phase_clean", np.nan)
            conf = rr.get("phase_confidence", np.nan)
        else:
            ph = np.nan
            conf = np.nan

        if pd.notna(ph):
            ph = int(ph)
            color = PHASE_COLORS.get(ph, (255,255,255))
            label = PHASE_NAMES.get(ph, str(ph))
        else:
            color = (150,150,150)
            label = "No phase"

        if last_pose is not None:
            draw_tasks_pose(frame, last_pose, color=color, line_thickness=3, radius=3)

        cv2.rectangle(frame, (10,10), (650,105), (0,0,0), -1)
        cv2.putText(frame, f"Phase: {label}", (25,45),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
        cv2.putText(frame, f"Frame: {fi}", (25,80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2, cv2.LINE_AA)

        if pd.notna(conf):
            cv2.putText(frame, f"Classifier confidence: {float(conf):.2f}", (245,80),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2, cv2.LINE_AA)

        writer.write(frame)
        fi += 1

    cap.release()
    writer.release()


In [ ]:
# ============================================================
# 5C. Run FAST MediaPipe phase + morphology analysis
# ============================================================
all_phase = []
all_morph_frames = []
phase_run_log = []

for i, r in video_table.iterrows():
    print("="*80)
    print(f"[{i+1}/{len(video_table)}] {r['group_label']} / {r['original_filename']}")
    print("source:", r["source_container"])
    print("path  :", r["source_relpath"])

    local_video = None

    try:
        local_video = materialize_video(r)

        phase_df_i, morph_df_i, pose_cache_i, perf_i = analyze_video_mediapipe(
            local_video,
            r["filename"],
            r["group"],
            r["group_label"],
            r["subject_id"],
        )

        phase_df_i["original_filename"] = r["original_filename"]
        phase_df_i["source_index"] = r["source_index"]
        phase_df_i["source_relpath"] = r["source_relpath"]

        if not morph_df_i.empty:
            morph_df_i["original_filename"] = r["original_filename"]
            morph_df_i["source_index"] = r["source_index"]
            morph_df_i["source_relpath"] = r["source_relpath"]

        all_phase.append(phase_df_i)
        if not morph_df_i.empty:
            all_morph_frames.append(morph_df_i)

        qc_path = None
        if GENERATE_QC_VIDEO:
            qc_name = (
                f"{r['group']}__s{int(r['source_index']):02d}__"
                f"{Path(r['filename']).stem}_phase_QC.mp4"
            )
            qc_path = PHASE_QC_DIR / qc_name
            render_phase_qc(
                local_video,
                phase_df_i,
                qc_path,
                pose_cache=pose_cache_i
            )

        infmask = phase_df_i["is_inference_frame"]
        detection_rate = (
            float(phase_df_i.loc[infmask, "detected"].fillna(False).mean())
            if infmask.any() else np.nan
        )

        phase_run_log.append({
            "filename": r["filename"],
            "original_filename": r["original_filename"],
            "group": r["group"],
            "source_index": r["source_index"],
            "source_container": r["source_container"],
            "source_relpath": r["source_relpath"],
            "status": "ok",
            "pose_detection_rate_on_inference_frames": detection_rate,
            "qc_video": str(qc_path) if qc_path is not None else "",
            **perf_i,
        })

    except Exception as e:
        import traceback
        traceback.print_exc()
        phase_run_log.append({
            "filename": r["filename"],
            "original_filename": r["original_filename"],
            "group": r["group"],
            "source_index": r["source_index"],
            "source_container": r["source_container"],
            "source_relpath": r["source_relpath"],
            "status": "failed",
            "error": repr(e),
        })

    finally:
        if local_video is not None:
            cleanup_materialized_video(r, local_video)

phase_frames_df = pd.concat(all_phase, ignore_index=True) if all_phase else pd.DataFrame()
morph_frames_df = pd.concat(all_morph_frames, ignore_index=True) if all_morph_frames else pd.DataFrame()
phase_run_log_df = pd.DataFrame(phase_run_log)

phase_frames_df.to_csv(OUTPUT_FOLDER / "phase_frame_predictions.csv", index=False)
phase_run_log_df.to_csv(OUTPUT_FOLDER / "phase_run_log.csv", index=False)

display(phase_run_log_df)

if not phase_run_log_df.empty and "elapsed_mediapipe_s" in phase_run_log_df.columns:
    oklog = phase_run_log_df[phase_run_log_df["status"] == "ok"]
    if not oklog.empty:
        print(f"Median MediaPipe time/video: {oklog['elapsed_mediapipe_s'].median():.1f} s")


In [ ]:

# ============================================================
# 5D. Phase boundaries and QC
# ============================================================
boundary_rows = []

if not phase_frames_df.empty:
    for filename, d in phase_frames_df.groupby("filename", sort=False):
        d = d.sort_values("frame").copy()
        valid = d["phase_clean"].notna()

        for ph in range(5):
            z = d[valid & (d["phase_clean"].round().astype("Int64") == ph)]
            if z.empty:
                continue

            boundary_rows.append({
                "filename": filename,
                "group": z["group"].iloc[0],
                "group_label": z["group_label"].iloc[0],
                "phase_id": ph,
                "phase_name": PHASE_NAMES[ph],
                "start_frame": int(z["frame"].min()),
                "end_frame": int(z["frame"].max()),
                "start_time_s": float(z["time_s"].min()),
                "end_time_s": float(z["time_s"].max()),
                "duration_s": float(z["time_s"].max() - z["time_s"].min()),
                "n_frames": int(len(z)),
            })

phase_boundaries_df = pd.DataFrame(boundary_rows)
phase_boundaries_df.to_csv(OUTPUT_FOLDER / "phase_boundaries.csv", index=False)

display(phase_boundaries_df.head(20))

if not phase_frames_df.empty:
    qc_summary = (
        phase_frames_df.groupby(["group","filename"])
        .agg(
            n_frames=("frame","size"),
            detection_rate=("detected","mean"),
            mean_confidence=("phase_confidence","mean"),
            n_clean_phases=("phase_clean","nunique"),
        )
        .reset_index()
    )
    qc_summary.to_csv(OUTPUT_FOLDER / "phase_detection_QC_summary.csv", index=False)
    display(qc_summary)



# 6. Morphology PC1

For each video, body-proportion ratios are summarized by the **median across valid frames**.

Primary PCA inputs:

- leg / trunk
- thigh / trunk
- shank / trunk
- arm / trunk
- forearm / upper-arm
- shoulder width / pelvis width

`head_trunk_ratio_proxy` is exported but **not included in the primary PCA**.


In [ ]:

# ============================================================
# 6A. Video-level morphology summaries
# ============================================================
if morph_frames_df.empty:
    raise RuntimeError("No morphology landmarks were extracted.")

morph_video_df = (
    morph_frames_df
    .groupby(["filename","subject_id","group","group_label"], as_index=False)
    .agg({
        **{c:"median" for c in MORPH_PRIMARY_FEATURES},
        "head_trunk_ratio_proxy":"median",
        "frame":"count",
    })
    .rename(columns={"frame":"n_valid_morph_frames"})
)

# Robust variability estimates (MAD) for QC
for feature in MORPH_PRIMARY_FEATURES:
    mad = (
        morph_frames_df
        .groupby("filename")[feature]
        .apply(lambda s: float(np.nanmedian(np.abs(s - np.nanmedian(s)))))
        .rename(feature + "_MAD")
        .reset_index()
    )
    morph_video_df = morph_video_df.merge(mad, on="filename", how="left")

morph_video_df.to_csv(
    OUTPUT_FOLDER / "morphology_body_proportions_raw.csv",
    index=False
)

display(morph_video_df)


In [ ]:

# ============================================================
# 6B. PCA -> Morphology PC1
# ============================================================
X = morph_video_df[MORPH_PRIMARY_FEATURES].copy()

# median imputation only for rare missing ratios
X = X.fillna(X.median())

scaler = StandardScaler()
Xz = scaler.fit_transform(X)

n_components = min(3, Xz.shape[0], Xz.shape[1])
if n_components < 1:
    raise RuntimeError("Too few subjects for PCA.")

pca = PCA(n_components=n_components)
scores = pca.fit_transform(Xz)

morphology_df = morph_video_df.copy()
morphology_df["Morphology_PC1"] = scores[:,0]
if scores.shape[1] > 1:
    morphology_df["Morphology_PC2"] = scores[:,1]

# Orient PC1 so that relatively longer legs vs trunk tend to be positive.
corr = np.corrcoef(
    morphology_df["Morphology_PC1"],
    morphology_df["leg_trunk_ratio"]
)[0,1]

if np.isfinite(corr) and corr < 0:
    morphology_df["Morphology_PC1"] *= -1
    pca.components_[0] *= -1

loadings_df = pd.DataFrame({
    "feature": MORPH_PRIMARY_FEATURES,
    "PC1_loading": pca.components_[0],
})

print("Explained variance ratio:", pca.explained_variance_ratio_)
display(loadings_df)
display(
    morphology_df[
        ["filename","group_label","Morphology_PC1"] + MORPH_PRIMARY_FEATURES
    ].sort_values("Morphology_PC1")
)

morphology_df.to_csv(OUTPUT_FOLDER / "morphology_with_PC1.csv", index=False)
loadings_df.to_csv(OUTPUT_FOLDER / "morphology_PC1_loadings.csv", index=False)


In [ ]:

# ============================================================
# 6C. Morphology QC figures
# ============================================================
FIG_DIR = OUTPUT_FOLDER / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(9,5))
for group, g in morphology_df.groupby("group_label"):
    y = np.zeros(len(g))
    plt.scatter(g["Morphology_PC1"], y, alpha=0.75, label=group)
plt.yticks([])
plt.xlabel("Morphology PC1")
plt.title("Video-derived body-proportion morphology axis")
plt.legend()
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.savefig(FIG_DIR / "Morphology_PC1_by_source_folder.png", dpi=180)
plt.show()
plt.close()

plt.figure(figsize=(7,5))
plt.barh(loadings_df["feature"], loadings_df["PC1_loading"])
plt.xlabel("PC1 loading")
plt.title("Morphology PC1 loadings")
plt.tight_layout()
plt.savefig(FIG_DIR / "Morphology_PC1_loadings.png", dpi=180)
plt.show()
plt.close()



# 7. OpenCap Monocular environment

This section follows the current **Slim** setup concept:

- Python 3.9
- OpenSim 4.4
- PyTorch 2.1.2 + CUDA 11.8
- mmcv-full 1.7.2
- ViTPose
- SLAHMR utilities

A separate micromamba environment is used so Colab's host Python remains available for MediaPipe/QC.

The notebook also:
- extracts the user's SMPL v1.1 ZIP,
- standardizes the three SMPL model filenames,
- downloads WHAM auxiliary files/checkpoints that do not require SMPL credentials.


In [ ]:
# ============================================================
# 7A. Clone repository + create isolated environment — v10
#     IMPORTANT: OpenCap modules are repository-local, so every
#     micromamba subprocess gets repo root on PYTHONPATH.
# ============================================================
import subprocess, os, json, textwrap

MICROMAMBA = "/usr/local/bin/micromamba"
MAMBA_ROOT = "/content/micromamba"
ENV_NAME = "opencapmono"

if not os.path.exists(MICROMAMBA):
    subprocess.run(
        "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest "
        "| tar -xvj bin/micromamba && "
        "mv bin/micromamba /usr/local/bin/micromamba",
        shell=True, check=True
    )

if not REPO_DIR.exists():
    subprocess.run([
        "git","clone","--recursive",
        "https://github.com/utahmobl/opencap-monocular.git",
        str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git","-C",str(REPO_DIR),"pull"], check=False)
    subprocess.run([
        "git","-C",str(REPO_DIR),
        "submodule","update","--init","--recursive"
    ], check=False)

env_list = subprocess.check_output(
    [MICROMAMBA,"env","list","-r",MAMBA_ROOT]
).decode()

if ENV_NAME not in env_list:
    subprocess.run([
        MICROMAMBA,"create","-y",
        "-r",MAMBA_ROOT,"-n",ENV_NAME,
        "-c","opensim-org","-c","conda-forge",
        "python=3.9",
        "opensim=4.4=py39np121",
        "numpy=1.21.6",
        "pip"
    ], check=True)

def _opencap_subprocess_env():
    env = os.environ.copy()
    repo_root = str(REPO_DIR)
    old = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = repo_root if not old else repo_root + os.pathsep + old
    return env

def mrun(args, check=False, cwd=None, capture=False):
    cmd = [MICROMAMBA, "run", "-r", MAMBA_ROOT, "-n", ENV_NAME] + list(args)

    # Match the repository/Docker execution context.
    run_cwd = str(REPO_DIR) if cwd is None else str(cwd)
    run_env = _opencap_subprocess_env()

    print(">", " ".join(map(str, cmd)))
    print("  cwd:", run_cwd)

    if capture:
        proc = subprocess.run(
            cmd,
            cwd=run_cwd,
            env=run_env,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )
        if proc.stdout:
            print(proc.stdout)
        if check and proc.returncode != 0:
            raise RuntimeError(
                f"Command failed with returncode={proc.returncode}\n"
                f"{proc.stdout[-12000:] if proc.stdout else ''}"
            )
        return proc

    return subprocess.run(
        cmd,
        check=check,
        cwd=run_cwd,
        env=run_env,
    )

print("Repository:", REPO_DIR)
print("Environment:", ENV_NAME)
print("Subprocess PYTHONPATH begins with:", REPO_DIR)


In [ ]:
# ============================================================
# 7B. Install OpenCap Mono dependencies — v10 compatibility fix
# ============================================================
import re

mrun([
    "pip","install",
    "torch==2.1.2+cu118",
    "torchvision==0.16.2+cu118",
    "torchaudio==2.1.2+cu118",
    "--index-url","https://download.pytorch.org/whl/cu118"
], check=True)

mrun(["pip","install","openmim","gdown"], check=True)

# OpenCap's current INSTALL_SLIM.md explicitly specifies mmcv-full==1.7.2.
mrun(["mim","install","mmcv-full==1.7.2"], check=True)

mrun([
    "pip","install","-r",
    str(REPO_DIR/"installation"/"requirements_slim.txt")
], check=True)

# ----------------------------------------------------------------
# ViTPose bundled mmpose compatibility shim
#
# The upstream ViTPose submodule contains mmpose 0.24.0 and an old
# hard-coded guard:
#     mmcv_maximum_version = '1.5.0'
#
# OpenCap's maintained slim environment now uses mmcv-full 1.7.2.
# Patch ONLY that version guard; do not alter pose-estimation code.
# The strict preflight in 7E will still fail if real API/runtime
# incompatibilities remain.
# ----------------------------------------------------------------
vit_mmpose_init = (
    REPO_DIR / "WHAM" / "third-party" / "ViTPose" /
    "mmpose" / "__init__.py"
)

if not vit_mmpose_init.exists():
    raise FileNotFoundError(
        f"ViTPose mmpose __init__.py not found: {vit_mmpose_init}"
    )

txt = vit_mmpose_init.read_text(encoding="utf-8")
patched = re.sub(
    r"mmcv_maximum_version\s*=\s*['\"]1\.5\.0['\"]",
    "mmcv_maximum_version = '1.7.2'",
    txt,
)

if patched == txt:
    if "mmcv_maximum_version = '1.7.2'" in txt:
        print("ViTPose MMCV guard already patched to 1.7.2")
    else:
        # Do not silently modify an unexpected upstream file.
        print(txt[:2500])
        raise RuntimeError(
            "Unexpected ViTPose mmpose version guard. "
            "Notebook intentionally stopped instead of applying a blind patch."
        )
else:
    vit_mmpose_init.write_text(patched, encoding="utf-8")
    print("Patched ViTPose mmpose MMCV upper bound: 1.5.0 -> 1.7.2")

mrun([
    "pip","install","-v","-e",
    str(REPO_DIR/"WHAM"/"third-party"/"ViTPose")
], check=True)

mrun([
    "pip","install","-e",
    str(REPO_DIR/"slahmr")
], check=True)

subprocess.run(
    f"cd '{REPO_DIR / 'installation'}' && make setup-intrinsics",
    shell=True,
    check=False
)

# Show the exact editable mmpose source and versions BEFORE model setup.
version_probe = r"""
import os, sys
import torch, mmcv, mmpose
print("cwd:", os.getcwd())
print("torch:", torch.__version__)
print("mmcv:", mmcv.__version__)
print("mmpose:", mmpose.__version__)
print("mmpose file:", mmpose.__file__)
print("repo root on sys.path:", r'__REPO__' in sys.path)
""".replace("__REPO__", str(REPO_DIR))

mrun(["python","-c",version_probe], check=True, capture=True)



# ----------------------------------------------------------------
# v12 batch-analysis patch
# OpenCap Slim intentionally omits PyTorch3D, but the current
# run_mono_standalone.py requests WHAM visualization=True.
# This caused the observed ModuleNotFoundError: pytorch3d.
# Numerical WHAM/OpenSim results do not require that visualization.
# ----------------------------------------------------------------
standalone_py = REPO_DIR / "run_mono_standalone.py"
standalone_txt = standalone_py.read_text(encoding="utf-8")

replacements = {
    '"visualize": True,': '"visualize": False,',
    '"plotting": True,': '"plotting": False,',
}

for old, new in replacements.items():
    if old in standalone_txt:
        standalone_txt = standalone_txt.replace(old, new)
        print("Patched run_mono_standalone:", old, "->", new)
    elif new in standalone_txt:
        print("Already patched:", new)
    else:
        print("WARNING: expected OpenCap setting not found:", old)

standalone_py.write_text(standalone_txt, encoding="utf-8")

verify_txt = standalone_py.read_text(encoding="utf-8")
if '"visualize": False,' not in verify_txt:
    raise RuntimeError("Could not disable WHAM visualization; stopping before batch run.")

print("OpenCap batch patch: WHAM visualization OFF, optimization plotting OFF")

# ----------------------------------------------------------------
# v15: make OpenCap ffmpeg trimming robust to odd video dimensions.
# libx264/yuv420p requires even width and height.
# ----------------------------------------------------------------
utils_optim_py = REPO_DIR / "utils" / "utils_optim.py"
u_txt = utils_optim_py.read_text(encoding="utf-8")
pad_expr = 'vf_parts.append("pad=ceil(iw/2)*2:ceil(ih/2)*2")'

if pad_expr not in u_txt:
    old_block = '''    if imageScaleFactor is not None:
        vf_parts.append("scale=iw/{:.0f}:-1".format(1 / imageScaleFactor))
    vf = ",".join(vf_parts)
'''
    new_block = '''    if imageScaleFactor is not None:
        vf_parts.append("scale=iw/{:.0f}:-1".format(1 / imageScaleFactor))
    # H.264/yuv420p requires even dimensions.
    vf_parts.append("pad=ceil(iw/2)*2:ceil(ih/2)*2")
    vf = ",".join(vf_parts)
'''
    if old_block in u_txt:
        u_txt = u_txt.replace(old_block, new_block)
        utils_optim_py.write_text(u_txt, encoding="utf-8")
        print("Patched ffmpeg trim: pad odd dimensions to even.")
    else:
        raise RuntimeError("Could not locate rewriteVideos filter block for even-dimension patch.")
else:
    print("ffmpeg even-dimension patch already present.")


print("Dependency installation + ViTPose compatibility setup completed.")


In [ ]:
# ============================================================
# 7C. SMPL ZIP AUTO-DETECT + MINIMAL EXTRACTION
#     v9: no interactive upload wait, no full ZIP extraction
# ============================================================
import os
import zipfile
import shutil
from pathlib import Path

def _normalize_drive_path(p):
    """Accept both 'content/drive/...' and '/content/drive/...'."""
    if p is None:
        return None
    txt = str(p).strip()
    if txt.startswith("content/"):
        txt = "/" + txt
    return Path(txt)

def _inspect_smpl_zip(zip_path):
    """
    Find the official SMPL Female / Neutral / Male PKLs inside a ZIP,
    regardless of their internal folder depth.
    """
    zip_path = Path(zip_path)

    with zipfile.ZipFile(zip_path, "r") as zf:
        members = [
            n for n in zf.namelist()
            if (not n.endswith("/")) and n.lower().endswith(".pkl")
        ]

    def pick(kind):
        for member in members:
            name = Path(member).name.lower()

            if kind == "female" and (
                "basicmodel_f_" in name or "smpl_female" in name
            ):
                return member

            if kind == "neutral" and (
                "basicmodel_neutral" in name or "smpl_neutral" in name
            ):
                return member

            if kind == "male" and (
                "basicmodel_m_" in name or "smpl_male" in name
            ):
                return member

        return None

    return {
        "SMPL_FEMALE.pkl": pick("female"),
        "SMPL_NEUTRAL.pkl": pick("neutral"),
        "SMPL_MALE.pkl": pick("male"),
    }

def _valid_smpl_zip(path, verbose=False):
    path = _normalize_drive_path(path)

    if path is None or not path.is_file():
        return False, None

    try:
        mapping = _inspect_smpl_zip(path)
        ok = all(mapping.values())

        if verbose:
            print("  ZIP:", path)
            for k, v in mapping.items():
                print(f"    {k:18s} <- {v}")

        return ok, mapping

    except zipfile.BadZipFile:
        return False, None
    except Exception as e:
        if verbose:
            print("  inspect failed:", repr(e))
        return False, None

def resolve_smpl_zip():
    """
    Fast exact-path checks first, then shallow globbing, then a bounded
    MyDrive fallback search. This prevents a slow full-Drive recursive scan.
    """
    candidates = []

    # 1) Explicit/common locations
    hints = list(globals().get("SMPL_ZIP_HINTS", []))
    hints += [
        globals().get("SMPL_ZIP_PATH", None),

        # User-reported path, normalized to an absolute Colab path
        Path("/content/drive/SMPL_python_v.1.1.0.zip"),

        # Normal Drive root after drive.mount('/content/drive')
        Path("/content/drive/MyDrive/SMPL_python_v.1.1.0.zip"),

        # Previous notebook location
        Path("/content/drive/MyDrive/OpenCapAssets/SMPL_python_v.1.1.0.zip"),
    ]

    for p in hints:
        if p is not None:
            candidates.append(_normalize_drive_path(p))

    # 2) Shallow matching, including filenames with '(1)' suffix.
    shallow_roots = [
        Path("/content/drive"),
        Path("/content/drive/MyDrive"),
        Path("/content/drive/MyDrive/OpenCapAssets"),
        globals().get(
            "ASSET_FOLDER",
            Path("/content/drive/MyDrive/OpenCapAssets")
        ),
    ]

    for root in shallow_roots:
        root = Path(root)
        if not root.exists():
            continue

        for pattern in [
            "SMPL_python_v*.zip",
            "*/SMPL_python_v*.zip",
        ]:
            try:
                candidates.extend(root.glob(pattern))
            except Exception:
                pass

    # Deduplicate while preserving order.
    unique = []
    seen = set()

    for p in candidates:
        p = _normalize_drive_path(p)
        key = str(p)

        if key not in seen:
            seen.add(key)
            unique.append(p)

    print("Searching for SMPL ZIP...")

    for p in unique:
        if p.exists():
            print(" check:", p)

        ok, mapping = _valid_smpl_zip(p)

        if ok:
            print("\nSMPL ZIP detected:", p)
            _valid_smpl_zip(p, verbose=True)
            return p, mapping

    # 3) Bounded fallback search under MyDrive.
    root = Path("/content/drive/MyDrive")

    if root.exists():
        root_depth = len(root.parts)
        scanned_dirs = 0
        MAX_DEPTH = 4
        MAX_DIRS = 4000

        print("\nCommon paths did not match; bounded MyDrive search...")

        for current, dirs, files in os.walk(root):
            current_p = Path(current)
            depth = len(current_p.parts) - root_depth
            scanned_dirs += 1

            dirs[:] = [
                d for d in dirs
                if not d.startswith(".")
                and d not in {"Trash", ".Trash", "__MACOSX"}
            ]

            if depth >= MAX_DEPTH:
                dirs[:] = []

            for fname in files:
                low = fname.lower()

                if not (
                    low.startswith("smpl_python_v")
                    and low.endswith(".zip")
                ):
                    continue

                p = current_p / fname
                ok, mapping = _valid_smpl_zip(p)

                if ok:
                    print("\nSMPL ZIP detected by bounded search:", p)
                    _valid_smpl_zip(p, verbose=True)
                    return p, mapping

            if scanned_dirs >= MAX_DIRS:
                print(
                    "Stopped bounded search after",
                    MAX_DIRS,
                    "directories."
                )
                break

    searched = "\n".join(" - " + str(p) for p in unique)

    raise FileNotFoundError(
        "SMPL ZIP could not be detected automatically.\n"
        "Checked common locations:\n"
        f"{searched}\n\n"
        "Recommended locations:\n"
        " - /content/drive/SMPL_python_v.1.1.0.zip\n"
        " - /content/drive/MyDrive/SMPL_python_v.1.1.0.zip\n"
        " - /content/drive/MyDrive/OpenCapAssets/SMPL_python_v.1.1.0.zip\n"
        "v9 does not wait for interactive files.upload()."
    )

# ------------------------------------------------------------
# Resolve and validate the ZIP
# ------------------------------------------------------------
SMPL_ZIP_PATH, smpl_member_map = resolve_smpl_zip()

print("\nUsing SMPL ZIP:", SMPL_ZIP_PATH)
print(
    "ZIP size:",
    round(SMPL_ZIP_PATH.stat().st_size / 1024**2, 1),
    "MB"
)

# ------------------------------------------------------------
# Extract ONLY the three required model PKLs
# ------------------------------------------------------------
smpl_extract = Path("/content/smpl_user_models")

if smpl_extract.exists():
    shutil.rmtree(smpl_extract)

smpl_extract.mkdir(parents=True, exist_ok=True)

smpl_map = {}

with zipfile.ZipFile(SMPL_ZIP_PATH, "r") as zf:
    for standardized_name, member in smpl_member_map.items():

        if member is None:
            raise RuntimeError(
                f"Required SMPL model not found for {standardized_name}"
            )

        dst = smpl_extract / standardized_name

        with zf.open(member, "r") as src, open(dst, "wb") as out:
            shutil.copyfileobj(src, out)

        smpl_map[standardized_name] = dst

print("\nExtracted required SMPL models:")

for name, path in smpl_map.items():
    print(
        f"  {name:18s}",
        f"{path.stat().st_size / 1024**2:.1f} MB",
        "<-",
        smpl_member_map[name]
    )

# WHAM expected location
smpl_target = (
    REPO_DIR
    / "WHAM"
    / "dataset"
    / "body_models"
    / "smpl"
)

smpl_target.mkdir(parents=True, exist_ok=True)

for name, src in smpl_map.items():
    shutil.copy2(src, smpl_target / name)

print("\nSMPL models copied to:", smpl_target)
print("SMPL auto-setup: OK")


In [ ]:
# ============================================================
# 7D. Download WHAM assets + required model-transfer data — v15
# ============================================================
WHAM_DIR = REPO_DIR / "WHAM"
(WHAM_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(WHAM_DIR / "dataset").mkdir(parents=True, exist_ok=True)

def gdown_if_missing(file_id, dest, min_bytes=1_000_000):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > min_bytes:
        print("exists:", dest)
        return dest
    mrun(
        ["gdown", f"https://drive.google.com/uc?id={file_id}", "-O", str(dest)],
        check=True
    )
    return dest

# ------------------------------------------------------------
# Official WHAM auxiliary SMPL-related data.
# Upstream WHAM fetch_demo_data.sh downloads Google Drive file:
# 1pbmzRbWGgae6noDIyQOnohzaVnX_csUZ
#
# v14 only checked J_regressor_h36m.npy, so an incomplete prior
# extraction could incorrectly skip model_transfer/smplx_to_smpl.pkl.
# ------------------------------------------------------------
dataset_dir = WHAM_DIR / "dataset"
model_transfer_target = dataset_dir / "model_transfer" / "smplx_to_smpl.pkl"
jreg_target = dataset_dir / "body_models" / "J_regressor_h36m.npy"

required_aux = [
    jreg_target,
    model_transfer_target,
]

missing_aux = [p for p in required_aux if not p.exists()]

if missing_aux:
    print("Missing WHAM auxiliary files:")
    for p in missing_aux:
        print("  ", p)

    body_tar = dataset_dir / "body_models.tar.gz"
    # Force refresh if the prior extracted tree was incomplete.
    if body_tar.exists():
        body_tar.unlink()

    gdown_if_missing(
        "1pbmzRbWGgae6noDIyQOnohzaVnX_csUZ",
        body_tar,
        min_bytes=100_000
    )

    subprocess.run(
        ["tar", "-xvf", str(body_tar), "-C", str(dataset_dir)],
        check=True
    )

    if body_tar.exists():
        body_tar.unlink()

# If upstream archive layout changes, find exact file and normalize
# to the path expected by utils/opensim/defaults.py.
if not model_transfer_target.exists():
    candidates = list(dataset_dir.rglob("smplx_to_smpl.pkl"))
    if candidates:
        model_transfer_target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(candidates[0], model_transfer_target)
        print("Normalized model-transfer path:", candidates[0], "->", model_transfer_target)

# Re-copy user-licensed SMPL files after auxiliary extraction.
smpl_target.mkdir(parents=True, exist_ok=True)
for name, src in smpl_map.items():
    shutil.copy2(src, smpl_target / name)

# Hard fail EARLY before a 5-6 minute WHAM run.
for p in required_aux:
    if not p.exists():
        raise FileNotFoundError(
            f"Required WHAM auxiliary file is still missing: {p}"
        )

print("WHAM auxiliary data: OK")
print("  model transfer:", model_transfer_target)

# Main WHAM / detector weights.
checkpoint_ids = {
    "wham_vit_w_3dpw.pth.tar": "1i7kt9RlCCCNEW2aYaDWVr-G778JkLNcB",
    "wham_vit_bedlam_w_3dpw.pth.tar": "19qkI-a6xuwob9_RFNSPWf1yWErwVVlks",
    "hmr2a.ckpt": "1J6l8teyZrL0zFzHhzkC7efRhU0ZJ5G9Y",
    "yolov8x.pt": "1zJ0KP23tXD42D47cw1Gs7zE2BA_V_ERo",
    "vitpose-h-multi-coco.pth": "1xyF7F3I7lWtdq82xmEPVQ5zl4HaasBso",
}
for fname, fid in checkpoint_ids.items():
    gdown_if_missing(fid, WHAM_DIR / "checkpoints" / fname)

# Current fork expects 133-keypoint whole-body ViTPose here.
wholebody_dir = WHAM_DIR / "checkpoints" / "vitpose+_huge_wholebody"
wholebody_dir.mkdir(parents=True, exist_ok=True)
WHOLEBODY_CKPT = wholebody_dir / "wholebody.pth"

if not WHOLEBODY_CKPT.exists() or WHOLEBODY_CKPT.stat().st_size < 1_000_000_000:
    print("Downloading ViTPose-H wholebody checkpoint (~2.55 GB)...")
    mrun(["pip", "install", "-q", "huggingface_hub"], check=True)

    download_script = r"""
from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil

src = hf_hub_download(
    repo_id="JunkyByte/easy_ViTPose",
    filename="torch/wholebody/vitpose-h-wholebody.pth",
)
dst = Path(r"__DEST__")
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst)
print("saved:", dst, "bytes:", dst.stat().st_size)
""".replace("__DEST__", str(WHOLEBODY_CKPT))

    mrun(["python", "-c", download_script], check=True, capture=True)

print("Whole-body ViTPose:", WHOLEBODY_CKPT, WHOLEBODY_CKPT.stat().st_size)
print("WHAM assets prepared.")


In [ ]:
# ============================================================
# 7E. STRICT OpenCap Mono environment + model preflight — v11
#     CUDA is checked explicitly before WHAM initialization.
# ============================================================
verify = r"""
import os
import sys
import traceback
from pathlib import Path

repo = Path(r"__REPO__").resolve()
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

print("preflight cwd:", os.getcwd())
print("sys.path[0:5]:", sys.path[:5])

failures = []

def check(name, fn):
    try:
        value = fn()
        print(f"[OK] {name}: {value}")
        return value
    except Exception as e:
        failures.append((name, repr(e), traceback.format_exc()))
        print(f"[FAILED] {name}: {e}")
        return None

torch = __import__("torch")
check("torch", lambda: torch.__version__)
check("opensim", lambda: __import__("opensim").__version__)
check("mmcv", lambda: __import__("mmcv").__version__)

def cuda_info():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is not available inside the opencapmono environment. "
            "This usually means the Colab runtime is CPU-only. "
            "Select Runtime -> Change runtime type -> T4 GPU and rerun from the top."
        )
    return (
        f"available=True | device={torch.cuda.get_device_name(0)} | "
        f"torch_cuda={torch.version.cuda}"
    )

cuda_ok = check("CUDA / NVIDIA GPU", cuda_info)

def mmpose_info():
    import mmpose
    return f"{mmpose.__version__} | {mmpose.__file__}"
check("mmpose", mmpose_info)

check(
    "optimization import",
    lambda: (__import__("optimization"), "OK")[1]
)
check(
    "WHAM.demo import",
    lambda: (__import__("WHAM.demo", fromlist=["initialize_wham"]), "OK")[1]
)

required_files = [
    repo / "WHAM" / "checkpoints" / "wham_vit_bedlam_w_3dpw.pth.tar",
    repo / "WHAM" / "checkpoints" / "hmr2a.ckpt",
    repo / "WHAM" / "checkpoints" / "yolov8x.pt",
    repo / "WHAM" / "checkpoints" / "vitpose+_huge_wholebody" / "wholebody.pth",
    repo / "WHAM" / "dataset" / "body_models" / "smpl" / "SMPL_NEUTRAL.pkl",
    repo / "WHAM" / "dataset" / "body_models" / "smpl" / "SMPL_MALE.pkl",
    repo / "WHAM" / "dataset" / "body_models" / "smpl" / "SMPL_FEMALE.pkl",
    repo / "WHAM" / "dataset" / "model_transfer" / "smplx_to_smpl.pkl",
]

def required_asset_status():
    missing = [str(p) for p in required_files if not p.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing OpenCap/WHAM assets:\n" + "\n".join(missing)
        )
    return "all required files present"

check("required model assets", required_asset_status)

def batch_patch_status():
    txt = (repo / "run_mono_standalone.py").read_text(encoding="utf-8")
    if '"visualize": False,' not in txt:
        raise RuntimeError("WHAM visualization is enabled; PyTorch3D would be required.")
    if '"plotting": False,' not in txt:
        raise RuntimeError("Optimization plotting is still enabled.")
    if 'pad=ceil(iw/2)*2:ceil(ih/2)*2' not in (repo / "utils" / "utils_optim.py").read_text(encoding="utf-8"):
        raise RuntimeError("Odd-dimension ffmpeg padding patch is missing.")
    return "WHAM visualization disabled; PyTorch3D not required; ffmpeg even-dimension patch OK"

check("OpenCap batch visualization patch", batch_patch_status)

# Only attempt WHAM initialization if CUDA passed.
if cuda_ok is not None:
    def init_wham():
        from WHAM.demo import initialize_wham
        initialize_wham()
        return "model initialization OK"
    check("WHAM initialize_wham()", init_wham)
else:
    print("[SKIP] WHAM initialize_wham(): CUDA preflight failed")

if failures:
    print("\n========== PREFLIGHT FAILURES ==========")
    for name, err, tb in failures:
        print(f"\n--- {name} ---")
        print(err)
        print(tb[-7000:])
    sys.exit(1)

print("\nAll OpenCap/WHAM preflight checks passed.")
""".replace("__REPO__", str(REPO_DIR))

preflight = mrun(
    ["python", "-c", verify],
    check=False,
    cwd=REPO_DIR,
    capture=True
)

if preflight.returncode != 0:
    raise RuntimeError(
        "OpenCap preflight failed. "
        "If CUDA/NVIDIA GPU is the failed item, switch Colab to a T4/A100/L4 GPU runtime, "
        "reconnect, and rerun from the top."
    )


### Interpreting common preflight messages

- **`Found no NVIDIA driver on your system`** → the current Colab runtime has no GPU attached. This is not a Python-package problem.
- **`DPVO is not properly installed. Only estimate in local coordinates!`** → expected for the Slim static-camera setup; not fatal.
- **`apex is not installed`** → warning from optional acceleration code; not the cause of the current failure.
- If GPU is enabled correctly, `nvidia-smi` should show a device and 7E should report `[OK] CUDA / NVIDIA GPU` before initializing WHAM.



# 8. Run OpenCap Monocular

The current repository runner supports:

- `video_path`
- `metadata_path`
- `calib_path`
- optional `intrinsics_path`
- `estimate_local_only=True`
- `session_id`
- `activity`

For pitching, `activity="other"` is used.

### Important
Because true height/mass and original camera intrinsics are unavailable, use these outputs primarily as **exploratory kinematics**.  
Do not present personalized kinetics, joint loads, or muscle forces as validated subject-specific measurements.


In [ ]:
# ============================================================
# 8A. Diagnostic OpenCap runner + approximate intrinsics
# ============================================================
WORK_DIR = Path("/content/pitching_opencap_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

runner_path = REPO_DIR / "_pitching_colab_runner.py"
runner_path.write_text(r"""
import argparse, json, sys, traceback
from run_mono_standalone import run_mono_standalone

p = argparse.ArgumentParser()
p.add_argument("--video", required=True)
p.add_argument("--metadata", required=True)
p.add_argument("--calib", default="")
p.add_argument("--intrinsics", required=True)
p.add_argument("--session_id", required=True)
p.add_argument("--rerun", type=int, default=1)
a = p.parse_args()

try:
    result = run_mono_standalone(
        video_path=a.video,
        metadata_path=a.metadata,
        calib_path=(a.calib or None),
        intrinsics_path=a.intrinsics,
        estimate_local_only=True,
        rerun=bool(a.rerun),
        session_id=a.session_id,
        activity="other",
    )

    print("__OPENCAP_RESULT_JSON__")
    print(json.dumps(result, ensure_ascii=False, default=str))

    # The standalone pipeline may return an "aborted" result instead of raising.
    if isinstance(result, dict) and result.get("aborted"):
        print("OpenCap aborted at:", result.get("aborted_stage"))
        print("Reason:", result.get("error"))
        sys.exit(2)

except Exception:
    print("\n========== OPENCAP EXCEPTION ==========")
    traceback.print_exc()
    sys.exit(1)
""", encoding="utf-8")

# local-only=True means WHAM does not use global SLAM calibration.
# v7 created an empty calib.txt; v8 avoids pretending that this is a valid calibration.
CALIB_PATH = Path(USER_CALIB_PATH) if USER_CALIB_PATH else None

def make_generic_intrinsics(video_path, out_path):
    """
    Create the intrinsics pickle INSIDE the opencapmono Python environment.

    Why:
    Colab host currently uses NumPy 2.x, while OpenCap uses NumPy 1.21.6.
    A NumPy-2 pickle may reference 'numpy._core' and cannot be unpickled by
    NumPy 1.21.6. Creating the pickle in the same environment that reads it
    avoids that incompatibility completely.
    """
    import cv2, numpy as np

    cap = cv2.VideoCapture(str(video_path))
    w = float(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = float(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    if not np.isfinite(w) or not np.isfinite(h) or w <= 0 or h <= 0:
        raise RuntimeError(f"Cannot read video size: {video_path}")

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    creator = r"""
import sys, pickle
from pathlib import Path
import numpy as np

w = float(sys.argv[1])
h = float(sys.argv[2])
out_path = Path(sys.argv[3])

f = w / (2.0 * np.tan(np.deg2rad(60.0) / 2.0))

K = np.array([
    [f,   0.0, w/2.0],
    [0.0, f,   h/2.0],
    [0.0, 0.0, 1.0],
], dtype=np.float64)

calib = {
    "intrinsicMat": K,
    "distortion": np.zeros((1,5), dtype=np.float64),
    "imageSize": np.array([[h],[w]], dtype=np.float64),
}

with open(out_path, "wb") as fobj:
    pickle.dump(calib, fobj, protocol=4)

print("intrinsics_created")
print("numpy_version:", np.__version__)
print("path:", out_path)
"""

    proc = mrun(
        ["python", "-c", creator, str(w), str(h), str(out_path)],
        check=True,
        cwd=REPO_DIR,
        capture=True,
    )

    if not out_path.exists() or out_path.stat().st_size < 100:
        raise RuntimeError(f"Intrinsics pickle was not created: {out_path}")

    return out_path


def verify_intrinsics_compatibility(intrinsics_path):
    """Load the pickle with OpenCap's own loader before expensive WHAM."""
    probe = r"""
import sys
from utils.utils_optim import load_intrinsics
p = sys.argv[1]
intr = load_intrinsics(p)
print("intrinsics_load_OK")
print(intr)
"""
    proc = mrun(
        ["python", "-c", probe, str(intrinsics_path)],
        check=False,
        cwd=REPO_DIR,
        capture=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(
            "Generated intrinsics is not compatible with the OpenCap environment.\n"
            + (proc.stdout or "")
        )
    return True


# One tiny preflight pickle catches NumPy pickle incompatibility BEFORE WHAM.
_intr_probe = WORK_DIR / "_intrinsics_compat_probe.pickle"
_creator_probe = r"""
import pickle
from pathlib import Path
import numpy as np
p = Path(r"__OUT__")
d = {
    "intrinsicMat": np.array([[1000.,0.,960.],[0.,1000.,540.],[0.,0.,1.]], dtype=np.float64),
    "distortion": np.zeros((1,5), dtype=np.float64),
    "imageSize": np.array([[1080.],[1920.]], dtype=np.float64),
}
with open(p, "wb") as f:
    pickle.dump(d, f, protocol=4)
print(np.__version__, p)
""".replace("__OUT__", str(_intr_probe))
mrun(["python","-c",_creator_probe], check=True, cwd=REPO_DIR, capture=True)
verify_intrinsics_compatibility(_intr_probe)
print("Intrinsics NumPy compatibility preflight: OK")

print("CALIB_PATH:", CALIB_PATH or "(None; local-only mode)")
print("Intrinsics: generated per video from resolution (~60 deg horizontal FOV)")

In [ ]:

# ============================================================
# 8B. Metadata YAMLs
# ============================================================
import yaml

for _, r in video_table.iterrows():
    stem = Path(r["filename"]).stem
    md = {
        "height_m": DEFAULT_HEIGHT_M,
        "mass_kg": DEFAULT_MASS_KG,
        "sex": str(r.get("sex", DEFAULT_SEX)),
    }
    with open(WORK_DIR / f"{r['group']}__{stem}.yaml", "w") as f:
        yaml.safe_dump(md, f)

print("Metadata YAMLs ready.")


In [ ]:
# ============================================================
# 8C. Batch OpenCap Mono — DIAGNOSTIC SAFE MODE
#     v12 captures traceback; WHAM visualization/PyTorch3D is disabled upstream.
# ============================================================
OPENCAP_LOG_DIR = OUTPUT_FOLDER / "opencap_logs"
OPENCAP_LOG_DIR.mkdir(parents=True, exist_ok=True)

STOP_AFTER_FIRST_OPENCAP_FAILURE = True

opencap_records = []

for i, r in video_table.iterrows():
    stem = Path(r["filename"]).stem
    session_id = f"{r['group']}__s{int(r['source_index']):02d}__{i:04d}__{stem}"

    print("\n" + "="*90)
    print(
        f"[OpenCap {i+1}/{len(video_table)}] "
        f"{r['group_label']} / {r['original_filename']}"
    )
    print("source:", r["source_container"])
    print("path  :", r["source_relpath"])

    local_video = None

    try:
        local_video = materialize_video(r)

        # Explicit per-video approximate intrinsics.
        intr_path = WORK_DIR / f"{stem}_intrinsics.pickle"
        make_generic_intrinsics(local_video, intr_path)
        verify_intrinsics_compatibility(intr_path)

        # Reuse a completed WHAM result when rerunning the same case.
        # This avoids repeating the ~6 min WHAM step after an optimization-only failure.
        pre_case_dir = REPO_DIR / "results" / session_id
        existing_wham = list(pre_case_dir.rglob("wham_output.pkl")) if pre_case_dir.exists() else []
        rerun_wham = bool(FORCE_RERUN_OPENCAP or not existing_wham)
        if existing_wham and not FORCE_RERUN_OPENCAP:
            print("Reusing existing WHAM output:", existing_wham[0])
        else:
            print("WHAM will run from scratch.")

        args = [
            "python", str(runner_path),
            "--video", str(local_video),
            "--metadata", str(WORK_DIR / f"{r['group']}__{stem}.yaml"),
            "--intrinsics", str(intr_path),
            "--session_id", session_id,
            "--rerun", "1" if rerun_wham else "0",
        ]

        if CALIB_PATH is not None:
            args += ["--calib", str(CALIB_PATH)]

        proc = mrun(args, check=False, capture=True)

        log_path = OPENCAP_LOG_DIR / f"{session_id}.log"
        log_path.write_text(proc.stdout or "", encoding="utf-8", errors="ignore")

        case_dir = REPO_DIR / "results" / session_id
        status = "ok" if proc.returncode == 0 else "failed"

        opencap_records.append({
            "filename": r["filename"],
            "original_filename": r["original_filename"],
            "subject_id": r["subject_id"],
            "group": r["group"],
            "group_label": r["group_label"],
            "source_index": r["source_index"],
            "source_container": r["source_container"],
            "source_relpath": r["source_relpath"],
            "session_id": session_id,
            "returncode": proc.returncode,
            "status": status,
            "case_dir": str(case_dir),
            "intrinsics_file": str(intr_path),
            "log_file": str(log_path),
        })

        if proc.returncode != 0:
            print("\nOpenCap FAILED. Log saved to:", log_path)
            print("\nLast log lines:")
            print("\n".join((proc.stdout or "").splitlines()[-100:]))

            if STOP_AFTER_FIRST_OPENCAP_FAILURE:
                print(
                    "\nSTOP_AFTER_FIRST_OPENCAP_FAILURE=True: "
                    "stopping here to avoid repeating the same failure for all 9 videos."
                )
                break

    except Exception as e:
        import traceback
        tb = traceback.format_exc()
        print(tb)

        log_path = OPENCAP_LOG_DIR / f"{session_id}_wrapper_error.log"
        log_path.write_text(tb, encoding="utf-8")

        opencap_records.append({
            "filename": r["filename"],
            "original_filename": r["original_filename"],
            "subject_id": r["subject_id"],
            "group": r["group"],
            "group_label": r["group_label"],
            "source_index": r["source_index"],
            "source_container": r["source_container"],
            "source_relpath": r["source_relpath"],
            "session_id": session_id,
            "returncode": -1,
            "status": "failed",
            "case_dir": "",
            "error": repr(e),
            "log_file": str(log_path),
        })

        if STOP_AFTER_FIRST_OPENCAP_FAILURE:
            break

    finally:
        if local_video is not None:
            cleanup_materialized_video(r, local_video)

opencap_run_df = pd.DataFrame(opencap_records)

opencap_run_df.to_csv(
    OUTPUT_FOLDER / "opencap_run_log.csv",
    index=False
)

display(opencap_run_df)

n_ok = int((opencap_run_df["status"] == "ok").sum()) if not opencap_run_df.empty else 0
n_fail = int((opencap_run_df["status"] == "failed").sum()) if not opencap_run_df.empty else 0
print(f"OpenCap success={n_ok}, failed={n_fail}")

if n_ok == 0:
    print(
        "\nNo OpenCap trial succeeded yet. "
        "Do NOT interpret the empty phase_wave_df downstream. "
        "Fix the first OpenCap log above, rerun 8C, then continue to 9A/10A."
    )


# 9. Extract OpenSim kinematics

The notebook searches each OpenCap result directory for the non-WHAM IK `.mot` output and extracts common pitching-relevant joint coordinates.

Priority:
- pelvis tilt
- lumbar extension / bending / rotation
- hip flexion
- knee angle
- ankle angle
- elbow flexion
- arm / shoulder flexion

The actual columns available depend on the OpenSim model/output.


In [ ]:

# ============================================================
# 9A. MOT utilities
# ============================================================
def read_mot(path):
    lines = Path(path).read_text(errors="ignore").splitlines()
    end = None
    for i, line in enumerate(lines):
        if line.strip().lower() == "endheader":
            end = i
            break
    if end is None:
        for i, line in enumerate(lines):
            if line.strip().lower().startswith("time"):
                end = i - 1
                break
    if end is None:
        raise ValueError(f"Cannot locate MOT header end: {path}")

    return pd.read_csv(path, sep=r"\s+", skiprows=end+1, engine="python")

def choose_ik_mot(case_dir):
    case_dir = Path(case_dir)
    files = list(case_dir.rglob("*.mot"))
    if not files:
        return None

    # Prefer IK outputs not marked WHAM.
    scored = []
    for p in files:
        s = str(p).lower()
        score = 0
        if "/ik/" in s.replace("\\","/"):
            score += 5
        if "wham" not in s:
            score += 3
        if "kinematic" in s or "ik" in p.name.lower():
            score += 1
        scored.append((score, p))
    return sorted(scored, key=lambda x:x[0], reverse=True)[0][1]

PATTERNS = {
    "pelvis_tilt": ["pelvis_tilt"],
    "pelvis_list": ["pelvis_list"],
    "pelvis_rotation": ["pelvis_rotation"],
    "lumbar_extension": ["lumbar_extension"],
    "lumbar_bending": ["lumbar_bending"],
    "lumbar_rotation": ["lumbar_rotation"],
    "hip_flexion_r": ["hip_flexion_r"],
    "hip_flexion_l": ["hip_flexion_l"],
    "knee_angle_r": ["knee_angle_r", "knee_flexion_r"],
    "knee_angle_l": ["knee_angle_l", "knee_flexion_l"],
    "ankle_angle_r": ["ankle_angle_r"],
    "ankle_angle_l": ["ankle_angle_l"],
    "elbow_flex_r": ["elbow_flex_r", "elbow_flexion_r"],
    "elbow_flex_l": ["elbow_flex_l", "elbow_flexion_l"],
    "arm_flex_r": ["arm_flex_r", "shoulder_flex_r"],
    "arm_flex_l": ["arm_flex_l", "shoulder_flex_l"],
}

def match_mot_columns(df):
    found = {}
    for key, pats in PATTERNS.items():
        for pat in pats:
            hits = [c for c in df.columns if pat in c.lower()]
            if hits:
                found[key] = hits[0]
                break
    return found

def resample_101(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    good = np.isfinite(x) & np.isfinite(y)
    if good.sum() < 3:
        return np.full(101, np.nan)
    x = x[good]; y = y[good]
    order = np.argsort(x)
    x = x[order]; y = y[order]
    xn = np.linspace(x.min(), x.max(), 101)
    return np.interp(xn, x, y)



# 10. Transfer MediaPipe phases to OpenSim time

The phase classifier and OpenCap do not necessarily have identical frame counts.

Therefore phase boundaries are transferred using **relative video time**:

\[
u = \frac{t}{T}
\]

and mapped onto the OpenSim `.mot` time range.

Each phase is then independently normalized to **0–100% (101 points)**.


In [ ]:
# ============================================================
# 10A. Extract whole-trial and phase-specific OpenSim waveforms
# ============================================================
if opencap_run_df.empty or not (opencap_run_df["status"] == "ok").any():
    print(
        "SKIP 10A: OpenCap has no successful trials. "
        "Check cell 8C and the saved opencap_logs/*.log first."
    )
    metrics_df = pd.DataFrame(columns=[
        "filename","subject_id","group","group_label","mot_file","n_matched_variables"
    ])
    whole_wave_df = pd.DataFrame(columns=[
        "filename","subject_id","group","group_label","variable","time_pct","value"
    ])
    phase_wave_df = pd.DataFrame(columns=[
        "filename","subject_id","group","group_label",
        "phase_id","phase_name","phase_pct","variable","value"
    ])
else:
    # ============================================================
    whole_wave_rows = []
    phase_wave_rows = []
    metric_rows = []

    for _, rr in opencap_run_df.iterrows():
        if rr["status"] != "ok":
            continue

        mot_path = choose_ik_mot(rr["case_dir"])
        if mot_path is None:
            print("No .mot found:", rr["filename"])
            continue

        try:
            mot = read_mot(mot_path)
        except Exception as e:
            print("MOT parse failed:", rr["filename"], e)
            continue

        time_col = "time" if "time" in mot.columns else mot.columns[0]
        mot_t = mot[time_col].astype(float).to_numpy()
        if len(mot_t) < 3:
            continue

        matched = match_mot_columns(mot)

        # MediaPipe timeline for this video
        pdv = phase_frames_df[phase_frames_df["filename"] == rr["filename"]].sort_values("frame")
        if pdv.empty:
            continue

        total_video_time = float(pdv["time_s"].max())
        if total_video_time <= 0:
            total_video_time = float(len(pdv) / max(float(pdv["fps"].iloc[0]), 1.0))

        trial_row = {
            "filename": rr["filename"],
            "subject_id": rr["subject_id"],
            "group": rr["group"],
            "group_label": rr["group_label"],
            "mot_file": str(mot_path),
            "n_matched_variables": len(matched),
        }

        for key, col in matched.items():
            y = pd.to_numeric(mot[col], errors="coerce").to_numpy(float)

            # smooth only for derivative summary, preserve raw for normalized output
            ys = y.copy()
            if np.isfinite(ys).sum() >= 9 and len(ys) >= 9:
                win = 9 if len(ys) >= 9 else max(3, len(ys)//2*2+1)
                try:
                    ys = savgol_filter(
                        pd.Series(ys).interpolate(limit_direction="both").to_numpy(),
                        win, 2
                    )
                except Exception:
                    pass

            with np.errstate(all="ignore"):
                vel = np.gradient(ys, mot_t)

            trial_row[f"{key}_min"] = float(np.nanmin(y))
            trial_row[f"{key}_max"] = float(np.nanmax(y))
            trial_row[f"{key}_rom"] = float(np.nanmax(y) - np.nanmin(y))
            trial_row[f"{key}_peak_abs_vel"] = float(np.nanmax(np.abs(vel)))
            trial_row[f"{key}_peak_vel_trial_pct"] = float(
                np.nanargmax(np.abs(vel)) / max(len(vel)-1,1) * 100
            )

            # Whole trial 101
            yw = resample_101(mot_t, y)
            for pct, val in enumerate(yw):
                whole_wave_rows.append({
                    "filename": rr["filename"],
                    "subject_id": rr["subject_id"],
                    "group": rr["group"],
                    "group_label": rr["group_label"],
                    "variable": key,
                    "time_pct": pct,
                    "value": val,
                })

            # Phase-specific mapping
            for ph in range(5):
                pp = pdv[pdv["phase_clean"].round() == ph]
                if pp.empty:
                    continue

                video_u0 = float(pp["time_s"].min() / total_video_time)
                video_u1 = float(pp["time_s"].max() / total_video_time)

                # Map relative video time into MOT time domain
                mot0 = mot_t.min() + video_u0 * (mot_t.max() - mot_t.min())
                mot1 = mot_t.min() + video_u1 * (mot_t.max() - mot_t.min())

                sel = (mot_t >= mot0) & (mot_t <= mot1)
                if sel.sum() < 3:
                    continue

                yp = resample_101(mot_t[sel], y[sel])

                for pct, val in enumerate(yp):
                    phase_wave_rows.append({
                        "filename": rr["filename"],
                        "subject_id": rr["subject_id"],
                        "group": rr["group"],
                        "group_label": rr["group_label"],
                        "phase_id": ph,
                        "phase_name": PHASE_NAMES[ph],
                        "phase_pct": pct,
                        "variable": key,
                        "value": val,
                    })

        metric_rows.append(trial_row)

    metrics_df = pd.DataFrame(metric_rows)
    whole_wave_df = pd.DataFrame(whole_wave_rows)
    phase_wave_df = pd.DataFrame(phase_wave_rows)

    metrics_df.to_csv(OUTPUT_FOLDER / "pitching_metrics_master.csv", index=False)
    whole_wave_df.to_csv(OUTPUT_FOLDER / "pitching_waveforms_wholetrial_101.csv", index=False)
    phase_wave_df.to_csv(OUTPUT_FOLDER / "phase_opensim_waveforms_101.csv", index=False)

    display(metrics_df.head())
    print("Whole-trial rows:", len(whole_wave_df))
    print("Phase-specific rows:", len(phase_wave_df))

In [ ]:
# ============================================================
# 10B. Merge Morphology PC1 into phase-specific waveforms
# ============================================================
if phase_wave_df.empty:
    print("WARNING: phase_wave_df is empty. Please check if OpenCap Mono run (cell 8C) or IK extraction (cell 10A) completed successfully.")
    phase_morph_df = pd.DataFrame()
else:
    phase_morph_df = phase_wave_df.merge(
        morphology_df[
            ["filename","Morphology_PC1"] + MORPH_PRIMARY_FEATURES
        ],
        on="filename",
        how="inner"
    )

    phase_morph_df.to_csv(
        OUTPUT_FOLDER / "phase_waveforms_with_Morphology_PC1.csv",
        index=False
    )

    print("Videos with both OpenCap phase waveforms and Morphology PC1:",
          phase_morph_df["filename"].nunique())
    display(phase_morph_df.head())


# 11. Descriptive figures + continuous morphology trend

The source folders are shown descriptively.

The main continuous screening model is:

\[
Y(t) = \beta_0(t) + \beta_1(t)\cdot MorphologyPC1
\]

The p-values generated here are **uncorrected point-wise screening values**, not formal SPM inference.

Use the exported long-format CSV for the final SPM analysis.


In [ ]:

# ============================================================
# 11A. Folder-wise mean phase waveforms
# ============================================================
if not phase_wave_df.empty:
    for ph in range(5):
        for var in sorted(phase_wave_df["variable"].unique()):
            d = phase_wave_df[
                (phase_wave_df["phase_id"] == ph) &
                (phase_wave_df["variable"] == var)
            ]
            if d.empty:
                continue

            plt.figure(figsize=(9,5))

            for group_key in ["elementary","mixed","high"]:
                g = d[d["group"] == group_key]
                if g.empty:
                    continue

                m = g.groupby("phase_pct")["value"].mean()
                se = g.groupby("phase_pct")["value"].sem()
                x = m.index.to_numpy()

                plt.plot(x, m.values, label=GROUP_LABELS[group_key])
                plt.fill_between(x, m.values-se.values, m.values+se.values, alpha=0.15)

            plt.xlabel(f"{PHASE_NAMES[ph]} normalized time (%)")
            plt.ylabel(var)
            plt.title(f"{PHASE_NAMES[ph]} — {var}")
            plt.legend()
            plt.grid(alpha=0.25)
            plt.tight_layout()

            out = FIG_DIR / f"group__phase{ph}_{PHASE_NAMES[ph]}__{var}.png"
            plt.savefig(out, dpi=180)
            plt.close()

print("Group figures saved to:", FIG_DIR)


In [ ]:

# ============================================================
# 11B. Point-wise Morphology PC1 trend screening
# ============================================================
trend_rows = []

if not phase_morph_df.empty:
    for (ph, var, pct), d in phase_morph_df.groupby(
        ["phase_id","variable","phase_pct"]
    ):
        dd = d[["Morphology_PC1","value"]].dropna()

        if len(dd) < 5 or dd["Morphology_PC1"].nunique() < 3:
            continue

        lr = linregress(dd["Morphology_PC1"], dd["value"])

        trend_rows.append({
            "phase_id": int(ph),
            "phase_name": PHASE_NAMES[int(ph)],
            "variable": var,
            "phase_pct": int(pct),
            "n": len(dd),
            "slope_per_PC1": lr.slope,
            "r": lr.rvalue,
            "p_uncorrected": lr.pvalue,
        })

morph_trend_df = pd.DataFrame(trend_rows)
morph_trend_df.to_csv(
    OUTPUT_FOLDER / "phase_waveform_vs_Morphology_PC1_screening.csv",
    index=False
)

display(morph_trend_df.head())


In [ ]:

# ============================================================
# 11C. Morphology trend slope figures
# ============================================================
if not morph_trend_df.empty:
    for (ph, var), d in morph_trend_df.groupby(["phase_id","variable"]):
        d = d.sort_values("phase_pct")

        plt.figure(figsize=(9,4))
        plt.plot(d["phase_pct"], d["slope_per_PC1"])
        plt.axhline(0, linewidth=1)

        plt.xlabel(f"{PHASE_NAMES[int(ph)]} normalized time (%)")
        plt.ylabel("Slope per Morphology PC1")
        plt.title(f"{PHASE_NAMES[int(ph)]}: {var} ~ Morphology PC1")
        plt.grid(alpha=0.25)
        plt.tight_layout()

        plt.savefig(
            FIG_DIR / f"morphtrend__phase{int(ph)}_{PHASE_NAMES[int(ph)]}__{var}.png",
            dpi=180
        )
        plt.close()

print("Morphology trend figures saved.")


In [ ]:
# ============================================================
# 12. Proximal-to-distal timing summary — SAFE
# ============================================================
import pandas as pd
import numpy as np

_REQUIRED_WHOLE_COLS = {"filename","variable","time_pct","value"}

if (
    whole_wave_df is None
    or whole_wave_df.empty
    or not _REQUIRED_WHOLE_COLS.issubset(set(whole_wave_df.columns))
):
    print(
        "SKIP 12: whole_wave_df has no usable OpenSim waveform data. "
        "Resolve section 8C/10A first; this is not a separate analysis error."
    )
    timing_df = pd.DataFrame(columns=[
        "filename","throwing_side",
        "lead_knee_peak_vel_pct","trunk_peak_vel_pct",
        "throwing_elbow_peak_vel_pct",
        "sequence_span_pct","sequence_violations",
        "Morphology_PC1"
    ])
else:
    side_map = video_table.set_index("filename")["throwing_side"].to_dict()
    timing_rows = []

    for filename, dfile in whole_wave_df.groupby("filename"):
        side = str(side_map.get(filename, "R")).upper()
        lead = "l" if side.startswith("R") else "r"
        arm  = "r" if side.startswith("R") else "l"

        targets = [
            (f"knee_angle_{lead}", "lead_knee"),
            ("lumbar_extension", "trunk"),
            (f"elbow_flex_{arm}", "throwing_elbow"),
        ]

        row = {"filename": filename, "throwing_side": side}
        timings = []

        for variable, label in targets:
            g = dfile[dfile["variable"] == variable].sort_values("time_pct")
            if len(g) < 5:
                continue

            x = g["time_pct"].to_numpy(float)
            y = g["value"].to_numpy(float)

            good = np.isfinite(x) & np.isfinite(y)
            if good.sum() < 5:
                continue

            x = x[good]
            y = y[good]
            v = np.gradient(y, x)
            peak_pct = float(x[np.nanargmax(np.abs(v))])

            row[label + "_peak_vel_pct"] = peak_pct
            timings.append((label, peak_pct))

        if len(timings) >= 2:
            vals = [x[1] for x in timings]
            row["sequence_span_pct"] = max(vals) - min(vals)

            expected_order = ["lead_knee","trunk","throwing_elbow"]
            available = {k:v for k,v in timings}
            ordered = [available[k] for k in expected_order if k in available]
            row["sequence_violations"] = int(
                sum(ordered[j] > ordered[j+1] for j in range(len(ordered)-1))
            )

        timing_rows.append(row)

    timing_df = pd.DataFrame(timing_rows)

    if not timing_df.empty and "filename" in timing_df.columns:
        timing_df = timing_df.merge(
            morphology_df[["filename","Morphology_PC1"]],
            on="filename",
            how="left"
        )

    timing_df.to_csv(
        OUTPUT_FOLDER / "kinematic_chain_timing_with_Morphology_PC1.csv",
        index=False
    )

    display(timing_df.head())



# 13. SPM-ready export

The key output is:

`phase_waveforms_with_Morphology_PC1.csv`

Each row contains:

- subject/video
- source folder
- phase
- normalized phase time 0–100
- OpenSim variable
- waveform value
- Morphology PC1
- body-proportion ratios

Recommended primary analysis:

```text
SPM regression:
waveform(t) ~ Morphology_PC1
```

Run this separately for biologically interpretable phase-variable pairs, rather than testing every possible output indiscriminately.

Suggested first targets:
- Stride: lead-knee angle / angular velocity-derived timing
- ArmCocking: lumbar / trunk variables
- ArmAcceleration: throwing-elbow variable


In [ ]:
# ============================================================
# 13A. Wide matrices for SPM1D — SAFE
# ============================================================
SPM_DIR = OUTPUT_FOLDER / "spm_ready"
SPM_DIR.mkdir(parents=True, exist_ok=True)

_REQUIRED_PHASE_MORPH_COLS = {
    "filename","phase_id","variable","phase_pct","value","Morphology_PC1"
}

if (
    phase_morph_df is None
    or phase_morph_df.empty
    or not _REQUIRED_PHASE_MORPH_COLS.issubset(set(phase_morph_df.columns))
):
    print(
        "SKIP 13A: phase_morph_df has no usable OpenCap phase-waveform data. "
        "Resolve section 8C/10A first."
    )
    spm_export_df = pd.DataFrame(columns=[
        "phase_id","phase_name","variable","n_subjects","file"
    ])
else:
    export_log = []

    for (ph, var), d in phase_morph_df.groupby(["phase_id","variable"]):
        mat = d.pivot_table(
            index="filename",
            columns="phase_pct",
            values="value",
            aggfunc="mean"
        )

        needed_cols = list(range(101))
        mat = mat.reindex(columns=needed_cols)

        meta = (
            morphology_df.set_index("filename")
            [["Morphology_PC1","group","group_label"]]
            .reindex(mat.index)
        )

        combined = pd.concat([meta, mat], axis=1)

        fname = f"phase{int(ph)}_{PHASE_NAMES[int(ph)]}__{var}__SPM.csv"
        combined.to_csv(SPM_DIR / fname, index=True)

        export_log.append({
            "phase_id": int(ph),
            "phase_name": PHASE_NAMES[int(ph)],
            "variable": var,
            "n_subjects": int(mat.notna().any(axis=1).sum()),
            "file": str(SPM_DIR / fname),
        })

    spm_export_df = pd.DataFrame(export_log)
    spm_export_df.to_csv(SPM_DIR / "SPM_export_index.csv", index=False)
    display(spm_export_df)


## v12 production-speed recommendation

First validate the 9-video test with:

```python
MEDIAPIPE_MODEL_VARIANT = "full"
TARGET_MEDIAPIPE_FPS = 30.0
GENERATE_QC_VIDEO = True
```

For the 400-video production run, after QC:

```python
TEST_MODE = False
MEDIAPIPE_MODEL_VARIANT = "full"
TARGET_MEDIAPIPE_FPS = 30.0
GENERATE_QC_VIDEO = False
```

For an additional speed experiment, set `MEDIAPIPE_MODEL_VARIANT = "lite"` and compare phase boundaries against Full on the 9-video test before applying Lite to all videos.



# 14. Final outputs

```text
OpenCapMono_Pitching_Output/
├── input_video_manifest.csv
├── phase_run_log.csv
├── phase_frame_predictions.csv
├── phase_boundaries.csv
├── phase_detection_QC_summary.csv
├── morphology_body_proportions_raw.csv
├── morphology_with_PC1.csv
├── morphology_PC1_loadings.csv
├── opencap_run_log.csv
├── pitching_metrics_master.csv
├── pitching_waveforms_wholetrial_101.csv
├── phase_opensim_waveforms_101.csv
├── phase_waveforms_with_Morphology_PC1.csv
├── phase_waveform_vs_Morphology_PC1_screening.csv
├── kinematic_chain_timing_with_Morphology_PC1.csv
├── qc_phase_videos/
├── figures/
└── spm_ready/
```

## Recommended workflow

### First run
```python
TEST_MODE = True
TEST_N_PER_GROUP = 3
```

Check:
- phase QC videos,
- phase coverage,
- pose detection rate,
- OpenCap success,
- available `.mot` columns,
- Morphology PC1 loadings.

### Full run
After QC is satisfactory:

```python
TEST_MODE = False
```

Then rerun from the folder-scan stage.

## Symposium-safe wording

Prefer:

> **AI-based monocular musculoskeletal reconstruction with a video-derived body-proportion morphology axis**

Avoid:

> “bone age estimated from video”  
> “validated 3D pitching kinetics”

unless separately validated.


### v6 input behavior

`TEST_MODE=True` means **3 videos per group**, not per source.

For a high-school group with two source archives, the test selector samples them round-robin whenever possible.

Large ZIP archives are not fully expanded. Each video is temporarily materialized only when needed.
